# Soil 16S phylogeny: Atacama ASVs

## Section 1: Story hook - Atacama Desert

The Atacama Desert is often described as the driest non-polar desert on Earth. Some places there have recorded zero rainfall across decades, yet microbial life persists in the soil.

Today we ask one question from three angles: **Who lives there, where are they abundant, and how are they related?**


## Section 2: Tree-thinking intro (mammals only)

A phylogenetic tree is a hypothesis about relatedness. Before looking at microbes, use familiar mammals to practice reading trees: dog, wolf, fox, bear, cat, and lion.

First question: if the same tree is drawn three ways, do the closest relatives change?


In [ ]:
import io
import json
import math
import random
import warnings
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, Markdown, display
from matplotlib import patches
from scipy import stats
try:
    from statsmodels.stats.multitest import multipletests
except ModuleNotFoundError:
    # Fallback only if a runtime lacks statsmodels. The intended path is statsmodels' fdr_bh.
    def multipletests(pvals, method="fdr_bh"):
        pvals = np.asarray(pvals, dtype=float)
        order = np.argsort(pvals)
        adjusted = np.empty_like(pvals)
        running = 1.0
        n = len(pvals)
        for rank_from_end, idx in enumerate(order[::-1], start=1):
            rank = n - rank_from_end + 1
            running = min(running, pvals[idx] * n / rank)
            adjusted[idx] = min(running, 1.0)
        return adjusted < 0.05, adjusted, None, None

warnings.filterwarnings("ignore", category=RuntimeWarning)

CACHE_FILES = {'goal2_atacama_sample_metadata.csv': 'sample_id,transect_name,site_name,depth,elevation,average_soil_relative_humidity,vegetation,percentcover\nBAQ1552.1.1,Baquedano,BAQ1552,1,1552,15.75,no,0\nBAQ2420.1.1,Baquedano,BAQ2420,1,2420,82.54,no,0\nBAQ2420.1.2,Baquedano,BAQ2420,2,2420,82.54,no,0\nBAQ2420.1.3,Baquedano,BAQ2420,3,2420,82.54,no,0\nBAQ2420.2,Baquedano,BAQ2420,2,2420,82.54,no,0\nBAQ2420.3,Baquedano,BAQ2420,2,2420,82.54,no,0\nBAQ2462.1,Baquedano,BAQ2462,2,2462,69.08,no,0\nBAQ2462.2,Baquedano,BAQ2462,2,2462,69.08,no,0\nBAQ2462.3,Baquedano,BAQ2462,2,2462,69.08,no,0\nBAQ2687.1,Baquedano,BAQ2687,2,2687,73.21,yes,0.1\nBAQ2687.2,Baquedano,BAQ2687,2,2687,73.21,yes,0.1\nBAQ2687.3,Baquedano,BAQ2687,2,2687,73.21,yes,0.1\nBAQ2838.1,Baquedano,BAQ2838,2,2838,44.74,no,0\nBAQ2838.2,Baquedano,BAQ2838,2,2838,44.74,no,0\nBAQ2838.3,Baquedano,BAQ2838,2,2838,44.74,no,0\nBAQ3473.1,Baquedano,BAQ3473,2,3473,82.05,yes,1.2\nBAQ3473.2,Baquedano,BAQ3473,2,3473,82.05,yes,1.2\nBAQ3473.3,Baquedano,BAQ3473,2,3473,82.05,yes,1.2\nBAQ4166.1.1,Baquedano,BAQ4166,1,4166,100,yes,7.1\nBAQ4166.1.2,Baquedano,BAQ4166,2,4166,100,yes,7.1\nBAQ4166.1.3,Baquedano,BAQ4166,3,4166,100,yes,7.1\nBAQ4166.2,Baquedano,BAQ4166,2,4166,100,yes,7.1\nBAQ4166.3,Baquedano,BAQ4166,2,4166,100,yes,7.1\nBAQ4697.1,Baquedano,BAQ4697,2,4697,,yes,0.1\nBAQ4697.2,Baquedano,BAQ4697,2,4697,,yes,0.1\nBAQ4697.3,Baquedano,BAQ4697,2,4697,,yes,0.1\nYUN1005.1.1,Yungay,YUN1005,1,1005,20.7,no,0\nYUN1005.3,Yungay,YUN1005,2,1005,20.7,no,0\nYUN1242.1,Yungay,YUN1242,2,1242,20.9,no,0\nYUN1242.2,Yungay,YUN1242,2,1242,20.9,no,0\nYUN1242.3,Yungay,YUN1242,2,1242,20.9,no,0\nYUN1609.1,Yungay,YUN1609,2,1609,17.18,no,0\nYUN2029.1,Yungay,YUN2029,2,2029,28.79,no,0\nYUN2029.2,Yungay,YUN2029,2,2029,28.79,no,0\nYUN2029.3,Yungay,YUN2029,2,2029,28.79,no,0\nYUN3008.1.3,Yungay,YUN3008,3,3008,70.89,no,0\nYUN3008.3,Yungay,YUN3008,2,3008,70.89,no,0\nYUN3153.2,Yungay,YUN3153,2,3153,59.69,no,0\nYUN3153.3,Yungay,YUN3153,2,3153,59.69,no,0\nYUN3184.2,Yungay,YUN3184,2,3184,26.97,no,0\nYUN3259.1.1,Yungay,YUN3259,1,3259,93.57,yes,2.4\nYUN3259.1.2,Yungay,YUN3259,2,3259,93.57,yes,2.4\nYUN3259.1.3,Yungay,YUN3259,3,3259,93.57,yes,2.4\nYUN3259.2,Yungay,YUN3259,2,3259,93.57,yes,2.4\nYUN3259.3,Yungay,YUN3259,2,3259,93.57,yes,2.4\nYUN3346.1,Yungay,YUN3346,2,3346,87.32,yes,0.01\nYUN3346.2,Yungay,YUN3346,2,3346,87.32,yes,0.01\nYUN3346.3,Yungay,YUN3346,2,3346,87.32,yes,0.01\nYUN3428.1,Yungay,YUN3428,2,3428,99.99,yes,8.8\nYUN3428.2,Yungay,YUN3428,2,3428,99.99,yes,8.8\nYUN3428.3,Yungay,YUN3428,2,3428,99.99,yes,8.8\nYUN3533.1.1,Yungay,YUN3533,1,3533,100,yes,8.6\nYUN3533.1.2,Yungay,YUN3533,2,3533,100,yes,8.6\nYUN3533.1.3,Yungay,YUN3533,3,3533,100,yes,8.6\nYUN3533.2,Yungay,YUN3533,2,3533,100,yes,8.6\nYUN3533.3,Yungay,YUN3533,2,3533,100,yes,8.6\nYUN3856.1.1,Yungay,YUN3856,1,3856,99.44,yes,3.1\nYUN3856.1.2,Yungay,YUN3856,2,3856,99.44,yes,3.1\nYUN3856.1.3,Yungay,YUN3856,3,3856,99.44,yes,3.1\nYUN3856.2,Yungay,YUN3856,2,3856,99.44,yes,3.1\nYUN3856.3,Yungay,YUN3856,2,3856,99.44,yes,3.1\n', 'goal2_atacama_counts_top50.csv': 'sample_id,Atacama_ASV_26,Atacama_ASV_13,Atacama_ASV_24,Atacama_ASV_01,Atacama_ASV_02,Atacama_ASV_03,Atacama_ASV_21,Atacama_ASV_15,Atacama_ASV_38,Atacama_ASV_04,Atacama_ASV_08,Atacama_ASV_16,Atacama_ASV_30,Atacama_ASV_31,Atacama_ASV_43,Atacama_ASV_07,Atacama_ASV_12,Atacama_ASV_17,Atacama_ASV_29,Atacama_ASV_32,Atacama_ASV_36,Atacama_ASV_37,Atacama_ASV_42,Atacama_ASV_46,Atacama_ASV_51,Atacama_ASV_52,Atacama_ASV_14,Atacama_ASV_18,Atacama_ASV_19,Atacama_ASV_22,Atacama_ASV_25,Atacama_ASV_35,Atacama_ASV_39,Atacama_ASV_40,Atacama_ASV_41,Atacama_ASV_44,Atacama_ASV_45,Atacama_ASV_47,Atacama_ASV_48,Atacama_ASV_49,Atacama_ASV_50,Atacama_ASV_53,Atacama_ASV_54,Atacama_ASV_05,Atacama_ASV_06,Atacama_ASV_23,Atacama_ASV_27,Atacama_ASV_28,Atacama_ASV_33,Atacama_ASV_34,total_reads\nBAQ1552.1.1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0\nBAQ2420.1.1,0,15,7,0,56,0,0,0,16,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,185\nBAQ2420.1.2,0,0,0,0,0,0,18,0,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,192,0,0,0,0,0,294\nBAQ2420.1.3,0,0,0,0,0,12,0,0,0,0,0,0,0,0,0,0,0,15,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,185,0,0,0,0,0,310\nBAQ2420.2,0,0,0,0,0,0,13,0,0,0,0,14,0,0,0,12,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4,0,0,0,0,0,0,18,0,296\nBAQ2420.3,0,0,16,0,19,0,0,0,3,0,30,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4,0,0,0,0,0,0,0,0,207\nBAQ2462.1,0,0,23,0,0,36,0,0,0,0,30,0,0,0,0,168,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,467\nBAQ2462.2,0,0,40,0,35,0,0,0,0,0,17,0,0,0,0,53,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,16,0,0,0,0,0,0,0,0,0,0,0,0,0,267\nBAQ2462.3,0,0,7,0,40,36,0,0,0,0,13,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,216\nBAQ2687.1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,39,0,0,0,0,0,0,0,0,0,0,0,0,30,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,266\nBAQ2687.2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,379\nBAQ2687.3,0,0,0,0,0,0,0,0,8,0,0,0,0,0,0,0,0,17,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,25,0,0,0,0,167\nBAQ2838.1,0,0,0,0,0,20,0,0,0,0,0,0,11,0,0,0,0,0,0,0,0,0,0,6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,138\nBAQ2838.2,0,0,0,0,0,5,0,0,0,0,0,0,0,0,0,0,0,22,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,43\nBAQ2838.3,0,0,0,0,41,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,82\nBAQ3473.1,0,0,0,0,0,0,0,0,0,254,0,0,0,0,0,0,0,0,0,0,15,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,15,0,0,0,0,0,0,0,0,0,0,0,0,0,489\nBAQ3473.2,0,0,0,193,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,18,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,270\nBAQ3473.3,0,25,0,0,0,0,0,0,0,190,0,0,0,0,0,0,0,0,0,0,23,0,0,0,9,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,529\nBAQ4166.1.1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,44,0,0,0,0,0,0,7,0,0,0,0,0,0,0,30,21,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,365\nBAQ4166.1.2,0,0,0,0,0,0,0,0,0,70,0,0,0,0,0,0,0,0,0,0,0,45,0,0,0,0,0,0,0,0,0,42,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,98,392\nBAQ4166.1.3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,21,0,0,0,0,0,0,0,0,35,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,390\nBAQ4166.2,13,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,23,0,55,0,0,0,0,0,0,0,0,0,0,0,0,0,14,0,18,0,0,0,0,0,0,0,0,0,0,0,0,0,335\nBAQ4166.3,8,221,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,37,0,0,0,0,0,0,5,0,0,0,0,0,0,0,0,46,0,0,16,0,0,0,0,0,0,0,0,0,0,0,40,481\nBAQ4697.1,15,0,0,0,0,0,0,64,0,33,0,0,0,0,0,0,155,0,0,44,0,0,18,0,0,0,0,0,0,0,0,0,49,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,494\nBAQ4697.2,18,0,0,0,0,0,0,139,0,0,0,0,0,0,0,0,36,0,0,45,0,0,19,0,0,0,0,0,0,0,0,0,0,0,0,0,0,15,0,0,0,0,0,0,0,0,0,0,0,0,399\nBAQ4697.3,20,7,0,0,0,0,0,70,0,32,0,0,0,0,0,0,183,0,0,63,0,0,26,0,0,0,0,0,0,0,0,0,65,0,0,0,0,0,0,0,12,0,0,0,0,0,0,0,0,0,664\nYUN1005.1.1,0,0,0,0,0,0,0,0,0,0,0,0,0,17,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,578\nYUN1005.3,0,0,0,0,0,0,0,0,0,0,0,0,0,15,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,278\nYUN1242.1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,376\nYUN1242.2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1\nYUN1242.3,0,0,16,0,36,53,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,489\nYUN1609.1,0,0,12,0,0,286,0,0,0,0,0,0,55,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,34,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,465\nYUN2029.1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0\nYUN2029.2,0,0,10,0,95,0,0,0,0,0,0,0,46,67,0,0,0,0,0,0,0,0,0,12,0,0,0,84,0,0,94,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,559\nYUN2029.3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1\nYUN3008.1.3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0\nYUN3008.3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1\nYUN3153.2,0,0,0,0,0,0,0,0,0,0,0,0,0,14,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,59,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,49,0,0,190\nYUN3153.3,0,5,4,0,46,0,0,0,0,0,0,0,20,41,0,0,0,0,0,0,0,0,0,0,0,0,0,20,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,52,0,0,334\nYUN3184.2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0\nYUN3259.1.1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0\nYUN3259.1.2,13,0,0,0,0,0,0,0,0,0,0,36,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,51,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,360\nYUN3259.1.3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,29,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,18,0,0,0,47\nYUN3259.2,19,20,0,0,0,0,0,0,11,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,60,0,0,53,0,182\nYUN3259.3,0,24,0,0,0,0,0,0,0,0,0,48,0,0,0,0,0,0,0,0,0,0,0,17,0,0,0,224,0,0,61,0,0,0,0,0,0,0,13,0,0,0,0,0,0,0,0,0,0,0,547\nYUN3346.1,0,0,0,0,0,0,0,0,0,0,0,114,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,12,0,0,0,0,0,0,0,0,405\nYUN3346.2,0,0,0,80,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,80\nYUN3346.3,0,0,0,0,0,0,0,0,0,0,0,40,23,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,164,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,373\nYUN3428.1,0,0,0,24,0,0,35,41,0,0,0,0,0,0,0,0,0,0,0,0,0,30,0,0,0,0,34,0,0,0,0,0,0,37,0,0,0,0,10,0,0,0,0,0,0,0,0,0,0,0,346\nYUN3428.2,0,0,0,0,0,0,35,0,0,0,0,50,0,0,0,0,0,0,0,0,0,0,0,0,0,13,0,0,27,0,0,0,0,0,0,20,0,0,0,0,0,0,0,0,0,0,0,0,0,0,466\nYUN3428.3,0,0,0,0,0,0,18,0,0,0,0,0,0,0,10,0,0,0,0,0,0,0,0,0,0,0,179,0,22,30,0,0,0,0,0,0,0,0,14,0,0,0,0,0,0,0,0,0,0,0,294\nYUN3533.1.1,9,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,319,0,0,0,0,0,0,491\nYUN3533.1.2,0,29,8,197,0,0,30,15,0,37,0,0,0,0,19,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,18,0,0,0,0,6,0,0,0,0,0,0,0,397\nYUN3533.1.3,0,0,0,187,0,0,0,0,0,0,27,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,49,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,13,0,0,0,387\nYUN3533.2,0,0,0,28,0,0,0,24,0,0,454,0,0,0,0,0,0,0,0,0,0,0,0,0,7,0,0,0,0,0,0,0,0,24,0,19,0,0,0,0,0,0,4,0,0,0,0,0,0,0,639\nYUN3533.3,0,24,0,47,0,0,0,23,0,0,0,0,0,0,20,0,0,0,0,0,0,0,0,0,14,0,0,0,0,0,0,0,0,0,0,37,0,0,0,0,0,0,0,0,0,0,0,0,0,0,434\nYUN3856.1.1,25,0,0,0,0,0,12,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,257,0,0,0,0,0,0,422\nYUN3856.1.2,11,0,0,258,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,21,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,23,0,0,8,0,0,0,0,0,0,0,500\nYUN3856.1.3,0,0,0,0,0,8,0,0,0,0,0,0,0,0,0,0,0,16,0,0,0,8,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,11,0,0,0,0,0,0,0,0,0,275\nYUN3856.2,0,27,0,203,0,0,0,0,15,0,0,0,0,0,9,0,0,0,0,44,0,0,0,0,0,6,0,0,0,0,0,0,0,0,0,0,0,0,0,13,17,0,0,0,0,0,0,0,0,0,500\nYUN3856.3,10,0,0,0,52,0,28,0,8,0,0,0,0,0,14,0,188,0,0,0,0,0,0,0,17,0,0,0,0,0,0,0,24,0,0,0,0,0,0,17,0,0,0,0,0,0,0,0,0,0,601\n', 'goal2_atacama_relative_abundance_top20.csv': 'sample_id,Atacama_ASV_01,Atacama_ASV_02,Atacama_ASV_03,Atacama_ASV_04,Atacama_ASV_05,Atacama_ASV_06,Atacama_ASV_07,Atacama_ASV_08,Atacama_ASV_11,Atacama_ASV_09,Atacama_ASV_10,Atacama_ASV_12,Atacama_ASV_13,Atacama_ASV_14,Atacama_ASV_15,Atacama_ASV_16,Atacama_ASV_17,Atacama_ASV_18,Atacama_ASV_19,Atacama_ASV_20,Other,total_reads\nBAQ1552.1.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0\nBAQ2420.1.1,0.0,30.27027,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.10811,0.0,0.0,0.0,0.0,0.0,0.0,0.0,61.62162,185\nBAQ2420.1.2,0.0,0.0,0.0,0.0,0.0,65.30612,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,34.69388,294\nBAQ2420.1.3,0.0,0.0,3.87097,0.0,0.0,59.67742,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.83871,0.0,0.0,0.0,31.6129,310\nBAQ2420.2,0.0,0.0,0.0,0.0,0.0,0.0,4.05405,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.72973,0.0,0.0,0.0,0.0,91.21622,296\nBAQ2420.3,0.0,9.17874,0.0,0.0,0.0,0.0,0.0,14.49275,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,76.3285,207\nBAQ2462.1,0.0,0.0,7.70878,0.0,0.0,0.0,35.9743,6.42398,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,49.89293,467\nBAQ2462.2,0.0,13.10861,0.0,0.0,0.0,0.0,19.85019,6.36704,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,60.67416,267\nBAQ2462.3,0.0,18.51852,16.66667,0.0,0.0,0.0,0.0,6.01852,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,58.7963,216\nBAQ2687.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,266\nBAQ2687.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,379\nBAQ2687.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.17964,0.0,0.0,0.0,89.82036,167\nBAQ2838.1,0.0,0.0,14.49275,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,85.50725,138\nBAQ2838.2,0.0,0.0,11.62791,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,51.16279,0.0,0.0,0.0,37.2093,43\nBAQ2838.3,0.0,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,50.0,82\nBAQ3473.1,0.0,0.0,0.0,51.94274,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,48.05726,489\nBAQ3473.2,71.48148,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,28.51852,270\nBAQ3473.3,0.0,0.0,0.0,35.91682,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.7259,0.0,0.0,0.0,0.0,0.0,0.0,0.0,59.35728,529\nBAQ4166.1.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,49.86301,50.13699,365\nBAQ4166.1.2,0.0,0.0,0.0,17.85714,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,82.14286,392\nBAQ4166.1.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,390\nBAQ4166.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,335\nBAQ4166.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,45.94595,0.0,0.0,0.0,0.0,0.0,0.0,0.0,54.05405,481\nBAQ4697.1,0.0,0.0,0.0,6.68016,0.0,0.0,0.0,0.0,0.0,0.0,0.0,31.37652,0.0,0.0,12.95547,0.0,0.0,0.0,0.0,0.0,48.98785,494\nBAQ4697.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.02256,0.0,0.0,34.83709,0.0,0.0,0.0,0.0,0.0,56.14035,399\nBAQ4697.3,0.0,0.0,0.0,4.81928,0.0,0.0,0.0,0.0,0.0,0.0,0.0,27.56024,1.05422,0.0,10.54217,0.0,0.0,0.0,0.0,0.0,56.0241,664\nYUN1005.1.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,578\nYUN1005.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,278\nYUN1242.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,376\nYUN1242.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1\nYUN1242.3,0.0,7.36196,10.83845,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,81.79959,489\nYUN1609.1,0.0,0.0,61.50538,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,38.49462,465\nYUN2029.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0\nYUN2029.2,0.0,16.99463,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,15.02683,0.0,0.0,67.97853,559\nYUN2029.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1\nYUN3008.1.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0\nYUN3008.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1\nYUN3153.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,190\nYUN3153.3,0.0,13.77246,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.49701,0.0,0.0,0.0,0.0,5.98802,0.0,0.0,78.74251,334\nYUN3184.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0\nYUN3259.1.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0\nYUN3259.1.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,0.0,0.0,0.0,0.0,90.0,360\nYUN3259.1.3,0.0,0.0,0.0,0.0,0.0,0.0,61.70213,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,38.29787,47\nYUN3259.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.98901,0.0,0.0,0.0,0.0,0.0,0.0,0.0,89.01099,182\nYUN3259.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.38757,0.0,0.0,8.77514,0.0,40.95064,0.0,0.0,45.88665,547\nYUN3346.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,28.14815,0.0,0.0,0.0,0.0,71.85185,405\nYUN3346.2,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,80\nYUN3346.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.72386,0.0,0.0,43.96783,0.0,45.30831,373\nYUN3428.1,6.93642,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.82659,11.84971,0.0,0.0,0.0,0.0,0.0,71.38728,346\nYUN3428.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.72961,0.0,0.0,5.79399,0.0,83.47639,466\nYUN3428.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,60.88435,0.0,0.0,0.0,0.0,7.48299,0.0,31.63265,294\nYUN3533.1.1,0.0,0.0,0.0,0.0,64.96945,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,35.03055,491\nYUN3533.1.2,49.62217,0.0,0.0,9.3199,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.30479,0.0,3.77834,0.0,0.0,0.0,0.0,0.0,29.97481,397\nYUN3533.1.3,48.32041,0.0,0.0,0.0,0.0,0.0,0.0,6.97674,0.0,0.0,0.0,0.0,0.0,12.6615,0.0,0.0,0.0,0.0,0.0,0.0,32.04134,387\nYUN3533.2,4.38185,0.0,0.0,0.0,0.0,0.0,0.0,71.04851,0.0,0.0,0.0,0.0,0.0,0.0,3.75587,0.0,0.0,0.0,0.0,0.0,20.81377,639\nYUN3533.3,10.82949,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.52995,0.0,5.29954,0.0,0.0,0.0,0.0,0.0,78.34101,434\nYUN3856.1.1,0.0,0.0,0.0,0.0,60.90047,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,39.09953,422\nYUN3856.1.2,51.6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,48.4,500\nYUN3856.1.3,0.0,0.0,2.90909,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.81818,0.0,0.0,0.0,91.27273,275\nYUN3856.2,40.6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,54.0,500\nYUN3856.3,0.0,8.65225,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,31.2812,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,60.06656,601\n', 'goal2_atacama_alpha_diversity.csv': 'sample_id,total_reads,observed_asvs,shannon_diversity\nBAQ1552.1.1,0,0,0.0\nBAQ2420.1.1,185,8,1.78\nBAQ2420.1.2,294,8,1.17438\nBAQ2420.1.3,310,10,1.48871\nBAQ2420.2,296,14,2.23713\nBAQ2420.3,207,11,2.06362\nBAQ2462.1,467,12,2.06846\nBAQ2462.2,267,12,2.27134\nBAQ2462.3,216,11,2.10769\nBAQ2687.1,266,9,2.08871\nBAQ2687.2,379,10,2.04318\nBAQ2687.3,167,10,2.18702\nBAQ2838.1,138,10,2.17046\nBAQ2838.2,43,4,1.20709\nBAQ2838.3,82,6,1.34959\nBAQ3473.1,489,11,1.66974\nBAQ3473.2,270,7,1.09648\nBAQ3473.3,529,16,2.27115\nBAQ4166.1.1,365,10,1.70418\nBAQ4166.1.2,392,10,2.05642\nBAQ4166.1.3,390,14,2.11929\nBAQ4166.2,335,13,2.44902\nBAQ4166.3,481,13,1.91312\nBAQ4697.1,494,14,2.26676\nBAQ4697.2,399,13,2.16252\nBAQ4697.3,664,20,2.53605\nYUN1005.1.1,578,17,2.17925\nYUN1005.3,278,8,1.72587\nYUN1242.1,376,8,1.72258\nYUN1242.2,1,1,-0.0\nYUN1242.3,489,14,2.23989\nYUN1609.1,465,8,1.34966\nYUN2029.1,0,0,0.0\nYUN2029.2,559,16,2.3881\nYUN2029.3,1,1,-0.0\nYUN3008.1.3,0,0,0.0\nYUN3008.3,1,1,-0.0\nYUN3153.2,190,7,1.67644\nYUN3153.3,334,13,2.3064\nYUN3184.2,0,0,0.0\nYUN3259.1.1,0,0,0.0\nYUN3259.1.2,360,9,1.84188\nYUN3259.1.3,47,2,0.6655\nYUN3259.2,182,9,1.75341\nYUN3259.3,547,13,1.99206\nYUN3346.1,405,15,2.30531\nYUN3346.2,80,1,-0.0\nYUN3346.3,373,10,1.71954\nYUN3428.1,346,13,2.39161\nYUN3428.2,466,17,2.63405\nYUN3428.3,294,9,1.42648\nYUN3533.1.1,491,9,1.29614\nYUN3533.1.2,397,12,1.82639\nYUN3533.1.3,387,11,1.7663\nYUN3533.2,639,12,1.24312\nYUN3533.3,434,19,2.72109\nYUN3856.1.1,422,9,1.40613\nYUN3856.1.2,500,15,1.91681\nYUN3856.1.3,275,14,2.10657\nYUN3856.2,500,18,2.27841\nYUN3856.3,601,25,2.66241\n', 'goal2_atacama_feature_key.csv': 'asv,qiime_feature_id,total_reads,prevalence_samples,mean_relative_abundance_percent,max_relative_abundance_percent,sequence_length,closest_taxonomic_match,phylum,family,genus,in_q_value_top50,in_abundance_top20,in_tree_top12,in_alignment_top8\nAtacama_ASV_01,409faa5f5353e543bf6d99125c7c0e83,1217,9,6.29134,100.0,227,wb1-P19,Proteobacteria,Nitrosococcaceae,wb1-P19,True,True,True,True\nAtacama_ASV_02,a36b38f754f6abd278aeb9dbc7696343,420,9,2.75176,50.0,227,Rubrobacter,Actinobacteriota,Rubrobacteriaceae,Rubrobacter,True,True,True,True\nAtacama_ASV_03,a7b877ae6d2f079a15b6b192a4425620,456,8,2.12492,61.50538,227,Nitriliruptoraceae,Actinobacteriota,Nitriliruptoraceae,,True,True,True,True\nAtacama_ASV_04,ffd60d684f32e6fd5b47fe90095f9d34,616,6,2.07436,51.94274,227,Actinobacteriota,Actinobacteriota,,,True,True,True,True\nAtacama_ASV_05,c434ee7f909f455fc2109ef1f741a2d4,576,2,2.06344,64.96945,227,Crossiella,Actinobacteriota,Pseudonocardiaceae,Crossiella,True,True,True,True\nAtacama_ASV_06,9279403ab31c9b2574b09cef10f08586,377,2,2.04891,65.30612,227,Ammoniphilus,Firmicutes,Paenibacillaceae,Ammoniphilus,True,True,True,True\nAtacama_ASV_07,ddbc32632c9632d1dd746f28721f3a9f,262,4,1.99313,61.70213,227,Ammoniphilus,Firmicutes,Paenibacillaceae,Ammoniphilus,True,True,True,True\nAtacama_ASV_08,1237d5925a7176fced9dda961a86c684,571,6,1.82504,71.04851,227,wb1-P19,Proteobacteria,Nitrosococcaceae,wb1-P19,True,True,True,True\nAtacama_ASV_09,ac0c1f67c014a92423a4aa57ab28a535,1,1,1.63934,100.0,227,Proteobacteria,Proteobacteria,,,False,True,True,False\nAtacama_ASV_10,d6d5efaea4b864401cf971d7d3de63d7,1,1,1.63934,100.0,227,Acidobacteriota,Acidobacteriota,,,False,True,True,False\nAtacama_ASV_11,e54fca6f1a506bfe90b8506cfa87ad1b,1,1,1.63934,100.0,227,Rubrobacter,Actinobacteriota,Rubrobacteriaceae,Rubrobacter,False,True,True,False\nAtacama_ASV_12,1694836fc379411d2b9aa087d68d571b,562,4,1.62689,31.37652,227,wb1-P19,Proteobacteria,Nitrosococcaceae,wb1-P19,True,True,True,False\nAtacama_ASV_13,ef3fdbe1dcde754d91130cde6a4b4d61,397,10,1.55643,45.94595,227,Gemmatimonadaceae,Gemmatimonadota,Gemmatimonadaceae,,True,True,False,False\nAtacama_ASV_14,a7fb4b2b79be3e851a32cc21d3a63e51,262,3,1.36676,60.88435,227,Actinobacteriota,Actinobacteriota,,,True,True,False,False\nAtacama_ASV_15,96cbccca68ad868a78bb0604e4a41cf5,376,7,1.36095,34.83709,227,wb1-P19,Proteobacteria,Nitrosococcaceae,wb1-P19,True,True,False,False\nAtacama_ASV_16,11f7b172e09b77715d3bb2175a40b409,302,6,1.19847,28.14815,227,wb1-P19,Proteobacteria,Nitrosococcaceae,wb1-P19,True,True,False,False\nAtacama_ASV_17,29b4b85879d0c908835a2c655cd82b76,70,4,1.18032,51.16279,227,Bacillus,Firmicutes,Bacillaceae,Bacillus,True,True,False,False\nAtacama_ASV_18,5987b97497dc2c6f7c3d690526ea34b0,328,3,1.01583,40.95064,227,wb1-P19,Proteobacteria,Nitrosococcaceae,wb1-P19,True,True,False,False\nAtacama_ASV_19,e043936a0f7ec9177eaaf79e4bbe8631,213,3,0.93844,43.96783,227,wb1-P19,Proteobacteria,Nitrosococcaceae,wb1-P19,True,True,False,False\nAtacama_ASV_20,dbaaec51eeca49cb19d683ec4ed2ea7a,182,1,0.81743,49.86301,227,Candidatus_Udaeobacter,Verrucomicrobiota,Chthoniobacteraceae,Candidatus_Udaeobacter,False,True,False,False\nAtacama_ASV_21,f6c10a04d57159c0d64d6bc30c677471,189,8,0.80856,10.11561,227,Sphingomonas,Proteobacteria,Sphingomonadaceae,Sphingomonas,True,False,False,False\nAtacama_ASV_22,c471b243ae160f3817bd392932447fce,123,3,0.79621,31.05263,227,Rubrobacter,Actinobacteriota,Rubrobacteriaceae,Rubrobacter,True,False,False,False\nAtacama_ASV_23,be2339ed842e86a8cbdb994dc01e84ec,85,2,0.78585,32.96703,227,Rubrobacter,Actinobacteriota,Rubrobacteriaceae,Rubrobacter,True,False,False,False\nAtacama_ASV_24,dc8a2f47b3d1dc2e1f5f805891976b29,143,10,0.74614,14.98127,227,Ralstonia,Proteobacteria,Burkholderiaceae,Ralstonia,True,False,False,False\nAtacama_ASV_25,c6a6705874a49360a01c2b3fd39fe900,206,3,0.69072,16.81574,227,67-14,Actinobacteriota,67-14,67-14,True,False,False,False\nAtacama_ASV_26,6b780e361cfc5f06def718518324bdcb,161,11,0.68484,10.43956,227,Arthrobacter,Actinobacteriota,Micrococcaceae,Arthrobacter,True,False,False,False\nAtacama_ASV_27,86b4c22b3bcd88d95151325f471af4c4,31,2,0.6829,38.29787,227,wb1-P19,Proteobacteria,Nitrosococcaceae,wb1-P19,True,False,False,False\nAtacama_ASV_28,c58d52926eceee08f400001aaf5dabb1,101,2,0.67801,25.78947,227,67-14,Actinobacteriota,67-14,67-14,True,False,False,False\nAtacama_ASV_29,476ca54c0b962e7b62e01fb0fc01890f,143,4,0.67663,14.66165,227,67-14,Actinobacteriota,67-14,67-14,True,False,False,False\nAtacama_ASV_30,5eee6273221f15d4e0cdc2c033f0dd72,155,5,0.65872,11.82796,227,Actinobacteriota,Actinobacteriota,,,True,False,False,False\nAtacama_ASV_31,6e4d034e4b967b745639f825dc73ebe6,154,5,0.65519,12.27545,227,0319-7L14,Actinobacteriota,0319-7L14,0319-7L14,True,False,False,False\nAtacama_ASV_32,352875c83f4adf659984d683666d2038,196,4,0.63071,11.2782,227,Gaiella,Actinobacteriota,Gaiellaceae,Gaiella,True,False,False,False\nAtacama_ASV_33,6466aa53adc18d74c66bfebb47d6c1fb,71,2,0.57708,29.12088,227,Bacillus,Firmicutes,Bacillaceae,Bacillus,True,False,False,False\nAtacama_ASV_34,f9c93ccfdaccc1f8111dea02735ff80a,138,2,0.54616,25.0,227,Gaiella,Actinobacteriota,Gaiellaceae,Gaiella,True,False,False,False\nAtacama_ASV_35,5c78314ff92e6fec9aa07acc1fa0dc24,107,3,0.50765,11.2782,227,Bradyrhizobium,Proteobacteria,Xanthobacteraceae,Bradyrhizobium,True,False,False,False\nAtacama_ASV_36,9b93b0042b0a3fba92a57224122cea67,111,4,0.5,16.41791,227,Mycobacterium,Actinobacteriota,Mycobacteriaceae,Mycobacterium,True,False,False,False\nAtacama_ASV_37,f2e9c4d09183997ce7cdab5a93686b4c,104,4,0.44687,11.47959,227,Gemmatimonadaceae,Gemmatimonadota,Gemmatimonadaceae,,True,False,False,False\nAtacama_ASV_38,a56b903521b8ab33a7878b35582803a8,65,7,0.43646,8.64865,227,Candidatus_Nitrososphaera,Crenarchaeota,Nitrososphaeraceae,Candidatus_Nitrososphaera,True,False,False,False\nAtacama_ASV_39,b65d67bf1cf90c3fcd9868edc2f35454,138,3,0.38855,9.91903,227,MB-A2-108,Actinobacteriota,MB-A2-108,MB-A2-108,True,False,False,False\nAtacama_ASV_40,e7e19b672a061e119437a7cf815db964,91,3,0.37162,10.69364,227,RB41,Acidobacteriota,Pyrinomonadaceae,RB41,True,False,False,False\nAtacama_ASV_41,0013c1743927ee19a962b903b8990896,81,3,0.31961,9.56341,227,Candidatus_Udaeobacter,Verrucomicrobiota,Chthoniobacteraceae,Candidatus_Udaeobacter,True,False,False,False\nAtacama_ASV_42,8d0ef18ec87e81bbb286975b9b0cb5b6,84,4,0.29026,5.38462,227,Actinobacteriota,Actinobacteriota,,,True,False,False,False\nAtacama_ASV_43,664c5245b6cbfea323173cf7123c585c,72,5,0.27746,4.78589,227,Acidobacteriota,Acidobacteriota,,,True,False,False,False\nAtacama_ASV_44,6c498e15165bcab4442f6e99135bcbb7,76,3,0.25886,8.52535,227,wb1-P19,Proteobacteria,Nitrosococcaceae,wb1-P19,True,False,False,False\nAtacama_ASV_45,fad7959733a200344fe5a2a9f6252f81,49,3,0.23661,5.99251,227,Blastococcus,Actinobacteriota,Geodermatophilaceae,Blastococcus,True,False,False,False\nAtacama_ASV_46,e64d870f57288ad0e8bd4da2d353bf90,38,4,0.21739,4.34783,228,TK10,Chloroflexi,TK10,TK10,True,False,False,False\nAtacama_ASV_47,88d17d59e9681436cc60ba2ef049d35d,49,3,0.19049,4.53401,227,Nitrososphaeraceae,Crenarchaeota,Nitrososphaeraceae,,True,False,False,False\nAtacama_ASV_48,36d3b99cbc044e1531dc21fb94159f72,37,3,0.1644,4.7619,227,Nitrososphaeraceae,Crenarchaeota,Nitrososphaeraceae,,True,False,False,False\nAtacama_ASV_49,c524ac41b7dc1cf61adcd13484ccc932,53,3,0.1644,4.6,227,67-14,Actinobacteriota,67-14,67-14,True,False,False,False\nAtacama_ASV_50,42db26c2c81f1d89d295d759f0490177,40,3,0.15094,4.0,227,wb1-P19,Proteobacteria,Nitrosococcaceae,wb1-P19,True,False,False,False\nAtacama_ASV_51,79fceab4210680594e1d67908b3f916d,47,4,0.1451,3.22581,227,Nitrososphaeraceae,Crenarchaeota,Nitrososphaeraceae,,True,False,False,False\nAtacama_ASV_52,4c8ff0ea98d2c0ebb486e33bd96c61f9,31,4,0.11389,2.7897,227,Povalibacter,Proteobacteria,Steroidobacteraceae,Povalibacter,True,False,False,False\nAtacama_ASV_53,4f52ef70709f6baf6ff9e9409f25d251,20,3,0.1024,2.96296,227,Gemmatimonadaceae,Gemmatimonadota,Gemmatimonadaceae,,True,False,False,False\nAtacama_ASV_54,89cb1ddf89dcf11d86f725dbcaa9a5ce,18,3,0.06127,1.6,227,Reyranella,Proteobacteria,Reyranellaceae,Reyranella,True,False,False,False\n', 'goal2_atacama_rep_seqs_top50_union.fasta': '>Atacama_ASV_01 qiime_feature_id=409faa5f5353e543bf6d99125c7c0e83\nAGCGTTAATCGGAATCACTGGGCGTAAAGGGCGCGTAGGCGGTTAGGTAAGTCGGATGTGAAAGCCCTGGGCTTAACCTG\nGGAATGGCATTCGAGACTGTCTATCTAGAGTCTGGTAGAGGGAAGTGGAATTTCCGGTGTAGCGGTGAAATGTGTAGATA\nTCGGAAGGAACACCAGTGGCGAAGGCGACTTCCTGGACCAAGACTGACGCTGAGGCGCGAAAGCGTG\n>Atacama_ASV_02 qiime_feature_id=a36b38f754f6abd278aeb9dbc7696343\nAGCGTTGTCCGGAATTATTGGGCGTAAAGAGCGTGTAGGCGGTTCGGTAAGTCTGCCGTGAAAACCTGGGGCTCAACCCC\nGGGCGTGCGGTGGATACTGCCGGGCTAGAGGATGGTAGAGGCGAGTGGAATTCCCGGTGTAGCGGTGAAATGCGCAGATA\nTCGGGAGGAACACCAGTAGCGAAGGCGGCTCGCTGGGCCATTCCTGACGCTGAGACGCGAAAGCTAG\n>Atacama_ASV_03 qiime_feature_id=a7b877ae6d2f079a15b6b192a4425620\nAGCGTTGTCCGGATTTATTGGGCGTAAAGAGCTCGTAGGCGGCCTGGTGAGTCGGGTGTGAAAGCCCGAGGCTCAACCTC\nGGAATTGCATTCGATACTGCTGGGCTTGAGGCAGGTAGGGGAGGATGGAATTCCCGGTGTAGCGGTGGAATGCGCAGATA\nTCGGGAGGAACACCTGCGGCGAAGGCGGTCCTCTGGGCCTGTCCTGACGCTGAGGAGCGAAAGCGTG\n>Atacama_ASV_04 qiime_feature_id=ffd60d684f32e6fd5b47fe90095f9d34\nAGCGTTGTCCGGAATCATTGGGCGTAAAGAGCGTGTAGGCGGTCCGGTAAGTCGGCTGTGAAAGTCCAGGGCTCAACCCT\nGGGATGCCGGTCGATACTGCCGGACTAGAGTTCGGAAGAGGCGAGTGGAATTCCCGGTGTAGCGGTGAAATGCGCAGATA\nTCGGGAGGAACACCTATGGCGAAGGCAGCTCGCTGGGACGTTACTGACGCTGAGACGCGAAAGCGTG\n>Atacama_ASV_05 qiime_feature_id=c434ee7f909f455fc2109ef1f741a2d4\nAGCGTTGTCCGGAATTATTGGGCGTAAAGAGCTCGTAGGCGGTCTGTCGCGTCGGCTGTGAAAACTCGGGGCTCAACTCC\nGAGCTTGCAGTCGATACGGGCAGGCTAGAGTTCGGCAGGGGAGACTGGAATTCCTGGTGTAGCGGTGAAATGCGCAGATA\nTCAGGAGGAACACCGGTGGCGAAGGCGGGTCTCTGGGCCGATACTGACGCTGAGGAGCGAAAGCGTG\n>Atacama_ASV_06 qiime_feature_id=9279403ab31c9b2574b09cef10f08586\nAGCGTTGTCCGGAATTATTGGGCGTAAAGCGCGCGCAGGCGGCTTACTAAGTCTGGTGTGAAAGCCCACGGCTCAACCGT\nGGAGGGCCATTGGAAACTGGTAGGCTTGAGTGCAGGAGAGGAGAGCGGAATTCCCGGTGTAGCGGTGAAATGCGTAGATA\nTCGGGAGGAACACCCGTGGCGAAGGCGGCTCTCTGGCCTGTAACTGACGCTGAGGCGCGAAAGCGTG\n>Atacama_ASV_07 qiime_feature_id=ddbc32632c9632d1dd746f28721f3a9f\nAGCGTTGTCCGGAATTATTGGGCGTAAAGCGCGCGCAGGCGGCTTACTAAGTCTGGTGTGAAAGCCCACGGCTCAACCGT\nGGAGGGCCATTGGAAACTGGTAAGCTTGAGTGCAGGAGAGGAGAGCGGAATTCCCGGTGTAGCGGTGAAATGCGTAGATA\nTCGGGAGGAACACCCGTGGCGAAGGCGGCTCTCTGGCCTGTAACTGACGCTGAGGCGCGAAAGCGTG\n>Atacama_ASV_08 qiime_feature_id=1237d5925a7176fced9dda961a86c684\nAGCGTTAATCGGAATTACTGGGCGTAAAGGGCGCGTAGGCGGTTGGGTAAGTCGGGTGTGAAAGCCCTGGGCTTAACCTG\nGGAATGGCATTCGAGACTACCTAGCTAGAGTCTGGTAGAGGGAAGTGGAATTTCCGGTGTAGCGGTGAAATGTGTAGATA\nTCGGAAGGAACACCAGTGGCGAAGGCGACTTCCTGGACCAAGACTGACGCTGAGGCGCGAAAGCGTG\n>Atacama_ASV_09 qiime_feature_id=ac0c1f67c014a92423a4aa57ab28a535\nAGCGTTGTTCGGATTTACTGGGCGTAAAGCGCACGTAGGCGGACTGTTAAGTCGGGGGTGAAATCCTGAGGCTCAACCTC\nAGAATTGCCTCCGATACTGGCGGTCCCGAGTACGGGAGAGGTGAGTGGAATTCCCAGTGTAGAGGTGAAATTCGTAGATA\nTTGGGAAGAACACCAGTGGCGAAGGCGGCTCACTGGCCCGTAACTGACGCTGAGGTGCGAAAGCGTG\n>Atacama_ASV_10 qiime_feature_id=d6d5efaea4b864401cf971d7d3de63d7\nAGCGTTGTTCGGAATTACTGGGCGTAAAGGGCTCGTAGGCGGCCAACTAAGTCACACGTGAAATCCCCCGGCTCAACCGG\nGGAACTGCGTGTGAGACTGGATGGCTTGAGTTTGGGAGAGGAATGCGGAATTCCAGGTGTAGCGGTGAAATGCGTAGATA\nTCTGGAGGAACACCGGTGGCGAAGGCGGCATTCTGGACCAACACTGACGCTGAGGAGCGAAAGCCAG\n>Atacama_ASV_11 qiime_feature_id=e54fca6f1a506bfe90b8506cfa87ad1b\nAGCGTTGTCCGGAATTATTGGGCGTAAAGAGCGTGTAGGCGGTTCGGTAAGTCTGTTGTGAAATCCTGGGGCTCAACCCC\nGGGCGTGCAACGGATACTGCCGGGCTAGAGGGTGGTAGAGGCAAGTGGAATTCCGAGTGTAGCGGTGAAATGCGCAGATA\nTTCGGAGGAACACCAGTAGCGAAGGCGGCTTGCTGGGCCACACCTGACGCTGAGACGCGAAAGCTAG\n>Atacama_ASV_12 qiime_feature_id=1694836fc379411d2b9aa087d68d571b\nAGCGTTAATCGGAATTACTGGGCGTAAAGGGCGCGTAGGCGGTGAAGTAAGTCGGGTGTGAAAGCCCCGGGCTCAACCTG\nGGAACTGCATCCGATACTGCTTCGCTAGAGTATGGTAGAGGGAAGCGGAATTCCGGGTGTAGCGGTGAAATGCGTAGATA\nTCCGGAGGAACACCAGTGGCGAAGGCGGCTTCCTGGACCAATACTGACGCTGAGGCGCGAAAGCGTG\n>Atacama_ASV_13 qiime_feature_id=ef3fdbe1dcde754d91130cde6a4b4d61\nAGCGTTGTCCGGAATCACTGGGCGTAAAGGGCGCGTAGGCGGCCTGATAAGTAGGGGGTGAAATCCTGCGGCTTAACCGC\nAGGGCTGCCTTCTAAACTGTCAGGCTCGAGCACAGTAGAGGCAGGTGGAATTCCCGGTGTAGCGGTGGAATGCGTAGAGA\nTCGGGAAGAACATCAGTGGCGAAGGCGGCCTGCTGGGCTGTTGCTGACGCTGAGGCGCGACAGCGTG\n>Atacama_ASV_14 qiime_feature_id=a7fb4b2b79be3e851a32cc21d3a63e51\nAGCGTTGTCCGGATTTATTGGGCGTAAAGAGCGTGTAGGCGGCCAGGTAGGTCTGCTGTGAAAACTCGAGGCTTAACCTC\nGAGATGTCGGCGGAAACCATCTGGCTAGAGTCCGGAAGAGGAGAATGGAATTCCCGGTGTAGCGGTGAAATGCGCAGATA\nTCGGGAAGAACACCCGTGGCGAAGGCGGTTCTCTGGGACGGTACTGACGCTGAGACGCGAAAGCGTG\n>Atacama_ASV_15 qiime_feature_id=96cbccca68ad868a78bb0604e4a41cf5\nAGCGTTAATCGGAATTACTGGGCGTAAAGGGCGCGTAGGCGGTGAAGTAAGTCGGGTGTGAAAGCCCCGGGCTCAACCTG\nGGAACTGCATTCGATACTGCTTCGCTAGAGTATGGTAGAGGGAAGCGGAATTCCGGGTGTAGCGGTGAAATGCGTAGATA\nTCCGGAGGAACACCAGTGGCGAAGGCGGCTTCCTGGACCAATACTGACGCTGAGGCGCGAAAGCGTG\n>Atacama_ASV_16 qiime_feature_id=11f7b172e09b77715d3bb2175a40b409\nAGCGTTAATCGGAATTACTGGGCGTAAAGGGCGCGTAGGCGGTGAAGTCAGTCGGGTGTGAAAGCCCCGGGCTCAACCTG\nGGAACTGCATCCGATACTGCTTCGCTAGAGTATGGTAGAGGGAAGCGGAATTCCGGGTGTAGCGGTGAAATGCGTAGATA\nTCCGGAGGAACACCAGTGGCGAAGGCGGCTTCCTGGACCAATACTGACGCTGAGGCGCGAAAGCGTG\n>Atacama_ASV_17 qiime_feature_id=29b4b85879d0c908835a2c655cd82b76\nAGCGTTGTCCGGAATTATTGGGCGTAAAGCGCGCGCAGGCGGTTCCTTAAGTCTGATGTGAAAGCCCACGGCTCAACCGT\nGGAGGGTCATTGGAAACTGGGGAACTTGAGTGCAGAAGAGAAGAGCGGAATTCCACGTGTAGCGGTGAAATGCGTAGAGA\nTGTGGAGGAACACCAGTGGCGAAGGCGGCTCTTTGGTCTGTAACTGACGCTGAGGCGCGAAAGCGTG\n>Atacama_ASV_18 qiime_feature_id=5987b97497dc2c6f7c3d690526ea34b0\nAGCGTTAATCGGAATTACTGGGCGTAAAGGGCGCGTAGGCGGTTGGGTAAGTCGGGTGTGAAAGCCCTGGGCTTAACCTG\nGGAATGGCATTCGAGACCACCTATCTAGAGTCTGGTAGAGGGAAGTGGAATTTCCGGTGTAGCGGTGAAATGTGTAGATA\nTCGGAAGGAACACCAGTGGCGAAGGCGACTTCCTGGACCAAGACTGACGCTGAGGCGCGAAAGCGTG\n>Atacama_ASV_19 qiime_feature_id=e043936a0f7ec9177eaaf79e4bbe8631\nAGCGTTAATCGGAATTACTGGGCGTAAAGGGCGCGTAGGCGGTGAAGTCAGTCGGGTGTGAAAGCCCCGGGCTCAACCTG\nGGAACGGCATCCGATACTGCTTCGCTAGAGTATGGTAGAGGGAAGCGGAATTCCGGGTGTAGCGGTGAAATGCGTAGATA\nTCCGGAGGAACACCAGTGGCGAAGGCGGCTTCCTGGACCAATACTGACGCTGAGGCGCGAAAGCGTG\n>Atacama_ASV_20 qiime_feature_id=dbaaec51eeca49cb19d683ec4ed2ea7a\nAGCGTTGTTCGGATTCATTGGGCGTAAAGGGTGTGTAGGTGGCGCCGTAAGTCGGGTGTGAAATCTCGGGGCTTAACTCC\nGAAACTGCATTCGATACTGCGGTGCTTGAGGACTGGAGAGGAGACTGGAATTCATGGTGTAGCAGTGAAATGCGTAGAGA\nTCATGAGGAAGACCAGTGGCGAAGGCGGGTCTCTGGACAGTTCCTGACACTGAGACACGAAGGCCAG\n>Atacama_ASV_21 qiime_feature_id=f6c10a04d57159c0d64d6bc30c677471\nAGCGTTGTTCGGAATTACTGGGCGTAAAGCGCACGTAGGCGGCTTTGTAAGTTAGAGGTGAAAGCCCGGGGCTCAACTCC\nGGAATTGCCTTTAAGACTGCATCGCTAGAATTGTGGAGAGGTGAGTGGAATTCCGAGTGTAGAGGTGAAATTCGTAGATA\nTTCGGAAGAACACCAGTGGCGAAGGCGACTCACTGGACACATATTGACGCTGAGGTGCGAAAGCGTG\n>Atacama_ASV_22 qiime_feature_id=c471b243ae160f3817bd392932447fce\nAGCGTTGTCCGGAATTATTGGGCGTAAAGAGCGTGTAGGCGGTTCGGTAAGTCTGCCGTGAAAACCTGGGGCTCAACCCC\nGGGCGTGCGGTGGATACTGCCGGGCTAGAGGGTGGTAGAGGCGAGTGGAATTCCCGGTGTAGCGGTGAAATGCGCAGATA\nTCGGGAGGAACACCAGTAGCGAAGGCGGCTCGCTGGGCCATTCCTGACGCTGAGACGCGAAAGCTAG\n>Atacama_ASV_23 qiime_feature_id=be2339ed842e86a8cbdb994dc01e84ec\nAGCGTTGTCCGGAATTATTGGGCGTAAAGAGCGTGTAGGCGGTTCGGTAAGTCTGCTGTGAAATCCTGGGGCTCAACCCC\nGGGCGTGCAGCGGATACTGCCGGGCTAGAGGATGGTAGAGGCGAGTGGAATTCCCGGTGTAGCGGTGAAATGCGCAGATA\nTCGGGAGGAACACCAGTAGCGAAGGCGGCTCGCTGGGCCATTCCTGACGCTGAGACGCGAAAGCTAG\n>Atacama_ASV_24 qiime_feature_id=dc8a2f47b3d1dc2e1f5f805891976b29\nAGCGTTAATCGGAATTACTGGGCGTAAAGCGTGCGCAGGCGGTTGTGCAAGACCGATGTGAAATCCCCGGGCTTAACCTG\nGGAATTGCATTGGTGACTGCACGGCTAGAGTGTGTCAGAGGGGGGTAGAATTCCACGTGTAGCAGTGAAATGCGTAGAGA\nTGTGGAGGAATACCGATGGCGAAGGCAGCCCCCTGGGATAACACTGACGCTCATGCACGAAAGCGTG\n>Atacama_ASV_25 qiime_feature_id=c6a6705874a49360a01c2b3fd39fe900\nAGCGTTGTCCGGAATCATTGGGCGTAAAGAGCGCGTAGGCGGTCCGGTAAGTCTGCCGTGAAAGCCAGGGGCTCAACCCT\nTGGATGCCGGTGGATACTGTCGGGCTAGAGTCCGGAAGAGGCGAGTGGAATTCCTGGTGTAGCGGTGAAATGCGCAGATA\nTCAGGAAGAACACCTATGGCGAAGGCAGCTCGCTGGGACGGAACTGACGCTGAGGCGCGAAAGCGTG\n>Atacama_ASV_26 qiime_feature_id=6b780e361cfc5f06def718518324bdcb\nAGCGTTATCCGGAATTATTGGGCGTAAAGAGCTCGTAGGCGGTTTGTCGCGTCTGCCGTGAAAGTCCGGGGCTCAACTCC\nGGATCTGCGGTGGGTACGGGCAGACTAGAGTGATGTAGGGGAGACTGGAATTCCTGGTGTAGCGGTGAAATGCGCAGATA\nTCAGGAGGAACACCGATGGCGAAGGCAGGTCTCTGGGCATTAACTGACGCTGAGGAGCGAAAGCATG\n>Atacama_ASV_27 qiime_feature_id=86b4c22b3bcd88d95151325f471af4c4\nAGCGTTAATCGGAATTACTGGGCGTAAAGGGCGCGTAGGCGGTGAAGTCAGTCGGGTGTGAAAGCCCCGGGCTCAACCTG\nGGAACTGCATTCGATACTGCTTCGCTAGAGTATGGTAGAGGGAAGCGGAATTCCGGGTGTAGCGGTGAAATGCGTAGATA\nTCCGGAGGAACACCAGTGGCGAAGGCGGCTTCCTGGACCAATACTGACGCTGAGGCGCGAAAGCGTG\n>Atacama_ASV_28 qiime_feature_id=c58d52926eceee08f400001aaf5dabb1\nAGCGTTGTCCGGAATCATTGGGCGTAAAGAGCGCGTAGGCGGTCCGGTAAGTCCATCGTGAAAGCCAGGGGCTCAACCCT\nTGGATGCCGGTGGATACTGTCGGGCTAGAGTCCGGAAGAGGCGAGTGGAATTCCTGGTGTAGCGGTGAAATGCGCAGATA\nTCAGGAAGAACACCTATGGCGAAGGCAGCTCGCTGGGACGGAACTGACGCTGAGGCGCGAAAGCGTG\n>Atacama_ASV_29 qiime_feature_id=476ca54c0b962e7b62e01fb0fc01890f\nAGCGTTGTCCGGAATTATTGGGCGTAAAGAGCGTGTAGGCGGTCCGGTAAGTCGGCTGTGAAAGTCCAGGGCTCAACCCT\nGGGATGCCGGTCGATACTGCCGGACTAGAGTTCGGAAGAGGCGAGTGGAATTCCCGGTGTAGCGGTGAAATGCGCAGATA\nTCGGGAGGAACACCTATGGCGAAGGCAGCTCGCTGGGACGTTACTGACGCTGAGACGCGAAAGCGTG\n>Atacama_ASV_30 qiime_feature_id=5eee6273221f15d4e0cdc2c033f0dd72\nAGCGTTGTCCGGATTTATTGGGCGTAAAGAGCGTGTAGGCGGCCGAGTAAGTCTGACGTGAAATCTGGAGGCTCAACCTC\nCAGCTGTCGTTGGAAACTATTCGGCTAGAGTCCGGAAGAGGAGAGTGGAATTCCCGGTGTAGCGGTGAAATGCGCAGATA\nTCGGGAAGAACACCCATGGCGAAGGCAGCTCTCTGGGACGGTACTGACGCTGAGACGCGAAAGCGTG\n>Atacama_ASV_31 qiime_feature_id=6e4d034e4b967b745639f825dc73ebe6\nAGCGTTGTCCGGATTTATTGGGCGTAAAGAGCTCGTAGGCGGCTGTTCGCGTCGGATGTGAAAGCTCAGAGCTCAACTCT\nGAGAGGCCATTCGATACGGGATAGCTAGAGGTAGGTAGGGGAGATCGGAATTCCTGGTGTAGCGGTGAAATGCGCAGATA\nTCAGGAGGAACACCGGTGGCGAAGGCGGATCTCTGGGCCTTACCTGACGCTGAGGAGCGAAAGCTGG\n>Atacama_ASV_32 qiime_feature_id=352875c83f4adf659984d683666d2038\nAGCGTTGTCCGGATTTATTGGGCGTAAAGAGCGTGTAGGCGGCTAGGTAGGTCCGTTGTGAAAACTCGAGGCTCAACCTC\nGAGACGTCGATGGAAACCATCTAGCTAGAGTCCGGAAGAGGAGAGTGGAATTCCTGGTGTAGCGGTGAAATGCGCAGATA\nTCAGGAAGAACACCCGTGGCGAAGGCGGCTCTCTGGTACGTGACTGACGCTGAGACGCGAAAGCGTG\n>Atacama_ASV_33 qiime_feature_id=6466aa53adc18d74c66bfebb47d6c1fb\nAGCGTTGTCCGGAATTATTGGGCGTAAAGCGCGCGCAGGCGGTCCTTTAAGTCTGATGTGAAAGCCCACGGCTCAACCGT\nGGAGGGTCATTGGAAACTGGGGGACTTGAGTACAGAAGAGGAAAGCGGAATTCCACGTGTAGCGGTGAAATGCGTAGAGA\nTGTGGAGGAACACCAGTGGCGAAGGCGGCTTTCTGGTCTGTAACTGACGCTGAGGCGCGAAAGCGTG\n>Atacama_ASV_34 qiime_feature_id=f9c93ccfdaccc1f8111dea02735ff80a\nAGCGTTGTCCGGATTTATTGGGCGTAAAGAGCGTGTAGGCGGCTAGATAGGTCCGTTGTGAAAACTCGAGGCTCAACCTC\nGAGACGTCGATGGAAACCATCTGGCTAGAGTCCGGAAGAGGAGAGTGGAATTCCTGGTGTAGCGGTGAAATGCGCAGATA\nTCAGGAAGAACACCCGTGGCGAAGGCGGCTCTCTGGTACGTGACTGACGCTGAGACGCGAAAGCGTG\n>Atacama_ASV_35 qiime_feature_id=5c78314ff92e6fec9aa07acc1fa0dc24\nAGCGTTGCTCGGAATCACTGGGCGTAAAGGGTGCGTAGGCGGGTCTTTAAGTCAGGGGTGAAATCCTGGAGCTCAACTCC\nAGAACTGCCTTTGATACTGAAGATCTTGAGTTCGGGAGAGGTGAGTGGAACTGCGAGTGTAGAGGTGAAATTCGTAGATA\nTTCGCAAGAACACCAGTGGCGAAGGCGGCTCACTGGCCCGATACTGACGCTGAGGCACGAAAGCGTG\n>Atacama_ASV_36 qiime_feature_id=9b93b0042b0a3fba92a57224122cea67\nAGCGTTGTCCGGAATTACTGGGCGTAAAGAGCTCGTAGGTGGTTTGTCGCGTTGTTCGTGAAAACTCACAGCTTAACTGT\nGGGCGTGCGGGCGATACGGGCAGACTGGAGTACTGCAGGGGAGACTGGAATTCCTGGTGTAGCGGTGGAATGCGCAGATA\nTCAGGAGGAACACCGGTGGCGAAGGCGGGTCTCTGGGCAGTAACTGACGCTGAGGAGCGAAAGCGTG\n>Atacama_ASV_37 qiime_feature_id=f2e9c4d09183997ce7cdab5a93686b4c\nAGCGTTGTCCGGAATCACTGGGCGTAAAGGGCGCGTAGGCGGCCTGATAAGTAGGGGGTGAAATCCTGCGGCTTAACCGC\nAGGGCTGCCTTCTAAACTGTCGGGCTCGAGCACAGTAGAGGCAGGTGGAATTCCCGGTGTAGCGGTGGAATGCGTAGAGA\nTCGGGAAGAACATCAGTGGCGAAGGCGGCCTGCTGGGCTGTTGCTGACGCTGAGGCGCGACAGCGTG\n>Atacama_ASV_38 qiime_feature_id=a56b903521b8ab33a7878b35582803a8\nAGTTGTCGGGACGATTATTGGGCCTAAAGCATCCGTAGCCTGTTCTGCAAGTCCTCCGTTAAATCCACCTGCTCAACGGA\nTGGGCTGCGGAGGATACCGCAGAGCTAGGAGGCGGGAGAGGCAAACGGTACTCAGTGGGTAGGGGTAAAATCCATTGATC\nTACTGAAGACCACCAGTGGCGAAGGCGGTTTGCCAGAACGCGCTCGACGGTGAGGGATGAAAGCTGG\n>Atacama_ASV_39 qiime_feature_id=b65d67bf1cf90c3fcd9868edc2f35454\nAGCGTTGTCCGGATTTATTGGGCGTAAAGAGCGCGTAGGCGGCTCGGAAAGTCGGTTGTGAAATCCCAGGGCTCAACCCC\nGGGACTGCGTCCGATACTGCCGAGCTAGAGGCAGGTAGGGGAGATCGGAATTCCTGGTGTAGCGGTGAAATGCGCAGATA\nTCAGGAGGAACACCGGTGGCGAAGGCGGATCTCTGGGCCTGTCCTGACGCTGAGGCGCGAAAGCTAG\n>Atacama_ASV_40 qiime_feature_id=e7e19b672a061e119437a7cf815db964\nAGCGTTGTTCGGATTTACTGGGCGTAAAGGGCGCGTAGGCGGCGCAACAAGTCACTTGTGAAATCTCCGGGCTTAACCCG\nGAGCGGCCAAGTGATACTGTCGTGCTAGAGTGCGGAAGGGGCTACTGGAATTCTCGGTGTAGCGGTGAAATGCGTAGATA\nTCGAGAGGAACACCTGCGGCGAAGGCGGGTAGCTGGGCCGACACTGACGCTGAGGCGCGAAAGCCAG\n>Atacama_ASV_41 qiime_feature_id=0013c1743927ee19a962b903b8990896\nAGCGTTGTTCGGATTCATTGGGCGTAAAGGGTGCGTAGGCGGCGCGGTAAGTCGGGTGTGAAATCTCGGGGCTTAACTCC\nGAAACTGCATTCGATACTACCGTGCTTGAGGACTGGAGAGGAGACTGGAATTTACGGTGTAGCGGTGAAATGCGTAGATA\nTCGTAAGGAAGACCAGTGGCGAAGGCGGGTCTCTGGACAGTTCCTGACGCTGAGGCACGAAGGCCAG\n>Atacama_ASV_42 qiime_feature_id=8d0ef18ec87e81bbb286975b9b0cb5b6\nAGCGTTGTCCGGATTTATTGGGCGTAAAGAGCGTGTAGGCGGCCAGGTAGGTCCGGTGTGAAAACTCGAGGCTTAACCTC\nGAGATGTCATCGGAAACCATCTGGCTAGAGTCCGGAAGAGGAGAGTGGAATTCCTGGTGTAGCGGTGAAATGCGCAGATA\nTCAGGAAGAACACCTATGGCGAAAGCAGCTCTCTGGGACGGTACTGACGCTGAGACGCGAAAGCGTG\n>Atacama_ASV_43 qiime_feature_id=664c5245b6cbfea323173cf7123c585c\nAGCGTTGTCCGGAATCACTGGGCGTAAAGGGCGCGTAGGTGGTTTGATAAGGGTGTGGTGAAAGTCCGGGGCTCAACCCC\nGGATCTGCCGTGCCGACTGTCAAACTCGAGGACTGTAGAGGCAGACGGAATTCCGGGTGTAGCGGTGGAATGCGTAGAGA\nTCCGGAGGAAGACCGGTGGCGAAGGCGGTCTGCTGGGCAGTTTCTGACACTGAGGCGCGACAGCGTG\n>Atacama_ASV_44 qiime_feature_id=6c498e15165bcab4442f6e99135bcbb7\nAGCGTTAATCGGAATTACTGGGCGTAAAGGGCGCGTAGGCGGTTGGGTAAGTCGGGTGTGAAAGCCCTGGGCTTAACCTG\nGGAACGGCATTCGAGACTACCTATCTAGAGTCTGGTAGAGGGAAGTGGAATTTCCGGTGTAGCGGTGAAATGTGTAGATA\nTCGGAAGGAACACCAGTGGCGAAGGCGACTTCCTGGACCAAGACTGACGCTGAGGCGCGAAAGCGTG\n>Atacama_ASV_45 qiime_feature_id=fad7959733a200344fe5a2a9f6252f81\nAGCGTTGTCCGGAATTATTGGGCGTAAAGAGCTCGTAGGCGGTTTGTTGCGTCGGCTGTGAAATCCCGAGGCTCAACCTC\nGGGTCTGCAGTCGATACGAGCAAACTAGAGTGTTGCAGGGGAGACTGGAATTCCTGGTGTAGCGGTGAAATGCGCAGATA\nTCAGGAGGAACACCGGTGGCGAAGGCGGGTCTCTGGGCAACAACTGACGCTGAGGAGCGAAAGCGTG\n>Atacama_ASV_46 qiime_feature_id=e64d870f57288ad0e8bd4da2d353bf90\nAGCGTTGTCCGGATTTATTGGGCGTAAAGCGCTCGCAGGCGGCTAGGCCAAGTCGTGCGTGAAATCTCTCGGCTCAACTG\nAGAGTTGTCGTGCGATACTGGTCAGCTCGAGGCCGGTAGGGGGAAGCGGAATTCCGGGTGTAGTGGTGGAATGCGTAGAT\nATCCGGAGGAACACCAGTGGCGAAGGCGGCTTCCTGGACCGGTTCTGACGCTCAGGAGCGAAAGCGTG\n>Atacama_ASV_47 qiime_feature_id=88d17d59e9681436cc60ba2ef049d35d\nAGTGGTCGGGACGATTATTGGGCCTAAAGCATCCGTAGCCGGTTTTACAAGTCCTCCGTTAAATCCAGCTGCTTAACAGA\nTGGGCTGCGGAGGATACTATAAGACTAGGAGGCAGGAGAGGCAAGCGGTACTCAGTGGGTAGGGGTAAAATCCGTTGATC\nCATTGAAGACCACCAGTGGCGAAGGCGGCTTGCCAGAATGCGCTCGACGGTGAGGGATGAAAGCTGG\n>Atacama_ASV_48 qiime_feature_id=36d3b99cbc044e1531dc21fb94159f72\nAGTGGTCGGGACGATTATTGGGCCTAAAGCATCCGTAGCCGGTCTTGCAAGTCTTCCGTTAAATCCAGCTGCTTAACAGA\nTGGGCTGCGGAGGATACTACAAGGCTAGGAGGCGGGAGAGGCAAGCGGTACTCAGTGGGTAGGGGTAAAATCCTCTGATC\nCATTGAAGACCACCAGTGGCGAAGGCGGCTTGCCAGAACGCGCTCGACGGTGAGGGATGAAAGCTGG\n>Atacama_ASV_49 qiime_feature_id=c524ac41b7dc1cf61adcd13484ccc932\nAGCGTTGTCCGGAATCATTGGGCGTAAAGAGCGTGTAGGCGGTCCGGTTAGTCGGCTGTGAAAGTCCAGGGCTCAACCCT\nGGGATGCCGGTCGATACTGCCGGACTAGAGTCCGGAAGAGGCAAGTGGAATTCCCGGTGTAGCGGTGAAATGCGCAGATA\nTCGGGAGGAACACCAATGGCGAAGGCAGCTTGCTGGGACGGTACTGACGCTGAGACGCGAAAGCGTG\n>Atacama_ASV_50 qiime_feature_id=42db26c2c81f1d89d295d759f0490177\nAGCGTTAATCGGAATTACTGGGCGTAAAGGGCGCGTAGGCGGTGAAGTAAGTCGGGTGTGAAAGCCCCGGGCTCAACCTG\nGGAACTGCCTCCGATACTGCTTCGCTAGAGTATGGTAGAGGGAAGCGGAATTCCGGGTGTAGCGGTGAAATGCGTAGATA\nTCCGGAGGAACACCAGTGGCGAAGGCGGCTTCCTGGACCAATACTGACGCTGAGGCGCGAAAGCGTG\n>Atacama_ASV_51 qiime_feature_id=79fceab4210680594e1d67908b3f916d\nAGTGGTCGGGACGATTATTGGGCCTAAAGCATCCGTAGCCGGTCATGCAAGTCCTCCGTTAAATCCACCTGCTTAACAGA\nTGGGCTGCGGAGGATACTACAAGGCTAGGAGGCGGAAGAGGCAAGCGGTACTCAGTGGGTAGGGGTAAAATCCTCTGATC\nCATTGAAGACCACCAGTGGCGAAGGCGGCTTGCCAATACGCGCTCGACGGTGAGGGATGAAAGCTGG\n>Atacama_ASV_52 qiime_feature_id=4c8ff0ea98d2c0ebb486e33bd96c61f9\nAGCGTTAATCGGAATTACTGGGCGTAAAGCGCGCGTAGGCGGCTTTGCAAGTCGGGTGTGAAATCCCCAGGCTTAACCTG\nGGAACTGCATTCGAGACTGCATTGCTAGAGTATGGGAGAGGGAAGTGGAATTTCCGGTGTAGCGGTGAAATGCGTAGATA\nTCGGAAGGAACATCAGTGGCGAAAGCGACTTCCTGGACCAATACTGACGCTCATGTGCGAAAGCGTG\n>Atacama_ASV_53 qiime_feature_id=4f52ef70709f6baf6ff9e9409f25d251\nAGCGTTGTCCGGAATCACTGGGCGTAAAGGGCGCGTAGGCGGCCTGGTAAGTAGGGGGTGAAATCCTGCGGCTTAACCGC\nAGGGCTGCCTTCTAAACTGTCAGGCTCGAGCACAGTAGAGGCAGGTGGAATTCCCGGTGTAGCGGTGGAATGCGTAGAGA\nTCGGGAAGAACATCAGTGGCGAAGGCGGCCTGCTGGGCTGTTGCTGACGCTGAGGCGCGACAGCGTG\n>Atacama_ASV_54 qiime_feature_id=89cb1ddf89dcf11d86f725dbcaa9a5ce\nAGCGTTGTTCGGAATTACTGGGCGTAAAGCGAGTGTAGGTGGTTGTCCAAGTTGGATGTGAAAGCCTTGAGCTCAACTCA\nAGAAATGCATTCAGGACTGGGCGGCTAGAGGACCGGAGAGGATAGTGGAATTCCCAGTGTAGTGGTGAAATACGTAGAGA\nTTGGGAAGAACACCAGTGGCGAAGGCGGCTATCTGGACGGTTTCTGACACTAAGACTCGAAAGCGTG\n', 'goal2_atacama_manifest.json': '{\n  "title": "Goal 2 Atacama soil ASV cache",\n  "source_table": "https://data.qiime2.org/2024.10/tutorials/chimera/atacama-table.qza",\n  "source_rep_seqs": "https://data.qiime2.org/2024.10/tutorials/chimera/atacama-rep-seqs.qza",\n  "source_metadata": "https://data.qiime2.org/2024.10/tutorials/atacama-soils/sample_metadata.tsv",\n  "source_context": "QIIME 2 2024.10 Atacama soil tutorial and q2-vsearch chimera tutorial artifacts.",\n  "source_resolution": {\n    "table": {\n      "path": "tmp\\\\atacama_qiime2_source\\\\atacama-table.qza",\n      "source": "local",\n      "sha256": "306a95685bf9687f61cdf0208a8e64c5e78d4ab461932ba3bccd2bd24c29e4bb"\n    },\n    "rep_seqs": {\n      "path": "tmp\\\\atacama_qiime2_source\\\\atacama-rep-seqs.qza",\n      "source": "local",\n      "sha256": "774570d071ba2e16886791c6b203d284d30258b13acf2792e39591768e0be02f"\n    },\n    "metadata": {\n      "path": "tmp\\\\atacama_qiime2_source\\\\sample_metadata.tsv",\n      "source": "local",\n      "sha256": "7cff810ad86a621ebc78a16b690255d839ea58e2622ad45a11a838d0f8c5b3bd"\n    }\n  },\n  "data_mode": "real_qiime2_artifacts_only_no_synthetic_counts",\n  "taxonomy_note": "Closest SILVA 138 515F/806R reference matches loaded from the local taxonomy cache.",\n  "sample_count": 61,\n  "feature_count_full_table": 401,\n  "present_threshold_samples": 7,\n  "asvs_at_or_above_10_percent_prevalence": 9,\n  "q_value_asvs": 50,\n  "abundance_asvs": 20,\n  "tree_asvs": 12,\n  "alignment_asvs": 8,\n  "scientific_note": "Taxonomy labels come from a local nearest-reference cache when available; otherwise they are left unassigned. Labels are closest matches, not species proof."\n}\n'}

OKABE_ITO = {
    "blue": "#0072B2",
    "orange": "#E69F00",
    "green": "#009E73",
    "pink": "#CC79A7",
    "sky": "#56B4E9",
    "vermillion": "#D55E00",
    "yellow": "#F0E442",
    "black": "#000000",
}
BASE_COLORS = {"A": "#56B4E9", "C": "#009E73", "G": "#E69F00", "T": "#CC79A7", "-": "#EEEEEE", "N": "#EEEEEE"}
SAMPLE_COLORS = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#56B4E9", "#D55E00", "#F0E442", "#000000"]

plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 180,
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.frameon": False,
})

def cache_text(name):
    local = Path("soil_16s_class_cache") / name
    if local.exists():
        return local.read_text(encoding="utf-8")
    return CACHE_FILES[name]

def read_cache_csv(name):
    return pd.read_csv(io.StringIO(cache_text(name)))

def parse_fasta(text):
    records = {}
    current = None
    chunks = []
    for line in text.splitlines():
        if not line.strip():
            continue
        if line.startswith(">"):
            if current is not None:
                records[current] = "".join(chunks)
            current = line[1:].split()[0]
            chunks = []
        else:
            chunks.append(line.strip().upper())
    if current is not None:
        records[current] = "".join(chunks)
    return records

def add_caption(fig, text):
    fig.text(0.01, -0.04, text, ha="left", va="top", fontsize=9, style="italic", color="#4D4D4D")

def clean_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    return ax

def styled_table(df, width_px=880):
    return (
        df.style.hide(axis="index")
        .set_table_styles([
            {"selector": "table", "props": [("border-collapse", "collapse"), ("width", f"{width_px}px"), ("font-size", "13px")]},
            {"selector": "th", "props": [("text-align", "left"), ("border-bottom", "1px solid #999"), ("padding", "6px 8px")]},
            {"selector": "td", "props": [("padding", "6px 8px"), ("border-bottom", "1px solid #e6e6e6")]},
            {"selector": "tbody tr:nth-child(odd)", "props": [("background-color", "#fafafa")]},
        ])
    )

def fmt_p(value):
    if pd.isna(value):
        return ""
    value = float(value)
    return f"{value:.2e}" if value < 0.001 else f"{value:.3f}"

def short_label(asv):
    return asv.replace("_", " ")

def sequence_distance(seq_a, seq_b):
    compared = 0
    differences = 0
    for a, b in zip(seq_a, seq_b):
        if a not in "ACGT" or b not in "ACGT":
            continue
        compared += 1
        differences += int(a != b)
    return differences / compared if compared else math.nan

metadata = read_cache_csv("goal2_atacama_sample_metadata.csv")
counts_top50 = read_cache_csv("goal2_atacama_counts_top50.csv")
relative_top20 = read_cache_csv("goal2_atacama_relative_abundance_top20.csv")
alpha_diversity = read_cache_csv("goal2_atacama_alpha_diversity.csv")
feature_key = read_cache_csv("goal2_atacama_feature_key.csv")
manifest = json.loads(cache_text("goal2_atacama_manifest.json"))
sequences = parse_fasta(cache_text("goal2_atacama_rep_seqs_top50_union.fasta"))

for col in ["average_soil_relative_humidity", "percentcover"]:
    metadata[col] = pd.to_numeric(metadata[col], errors="coerce")
for col in ["mean_relative_abundance_percent", "max_relative_abundance_percent", "prevalence_samples"]:
    feature_key[col] = pd.to_numeric(feature_key[col], errors="coerce")

feature_key = feature_key.sort_values("mean_relative_abundance_percent", ascending=False).reset_index(drop=True)
top20_asvs = feature_key.query("in_abundance_top20 == True")["asv"].tolist()
top12_asvs = feature_key.query("in_tree_top12 == True")["asv"].tolist()
top8_asvs = feature_key.query("in_alignment_top8 == True")["asv"].tolist()
q_asvs = feature_key.query("in_q_value_top50 == True")["asv"].tolist()

# ASV cascade - chosen for readability at each step:
#   top 50 by prevalence  -> q-value tests (statistical power)
#   top 20 by mean abundance -> abundance plots (readable bars)
#   top 12 by mean abundance -> UPGMA tree (readable tip labels)
#   top 8  by mean abundance -> alignment heatmap (readable bases)
# The 61-sample source table has fewer than 50 ASVs at >=10% prevalence,
# so the q-value section keeps the requested top-50 scale and interprets low-prevalence results cautiously.

asv_color = {asv: SAMPLE_COLORS[i % len(SAMPLE_COLORS)] for i, asv in enumerate(top20_asvs)}
asv_color["Other"] = "#BDBDBD"

@dataclass
class TreeNode:
    name: str | None = None
    left: object | None = None
    right: object | None = None
    left_length: float = 0.0
    right_length: float = 0.0

    @property
    def is_leaf(self):
        return self.name is not None

def mammal_tree(rotated=False):
    dog_wolf = TreeNode(left=TreeNode("dog"), right=TreeNode("wolf"), left_length=1.0, right_length=1.0)
    dog_wolf_fox = TreeNode(left=dog_wolf, right=TreeNode("fox"), left_length=1.0, right_length=2.0)
    cat_lion = TreeNode(left=TreeNode("cat"), right=TreeNode("lion"), left_length=2.0, right_length=2.0)
    bear_cat_lion = TreeNode(left=TreeNode("bear"), right=cat_lion, left_length=3.0, right_length=1.5)
    if rotated:
        dog_wolf = TreeNode(left=TreeNode("wolf"), right=TreeNode("dog"), left_length=1.0, right_length=1.0)
        dog_wolf_fox = TreeNode(left=TreeNode("fox"), right=dog_wolf, left_length=2.0, right_length=1.0)
        cat_lion = TreeNode(left=TreeNode("lion"), right=TreeNode("cat"), left_length=2.0, right_length=2.0)
        bear_cat_lion = TreeNode(left=cat_lion, right=TreeNode("bear"), left_length=1.5, right_length=3.0)
    return TreeNode(left=dog_wolf_fox, right=bear_cat_lion, left_length=1.0, right_length=0.5)

def assign_y(node, leaf_order):
    lookup = {name: i for i, name in enumerate(leaf_order)}
    ys = {}
    def walk(current):
        if current.is_leaf:
            ys[id(current)] = lookup[current.name]
            return ys[id(current)]
        left_y = walk(current.left)
        right_y = walk(current.right)
        ys[id(current)] = (left_y + right_y) / 2
        return ys[id(current)]
    walk(node)
    return ys

def draw_tree(ax, node, leaf_order, label_map=None, x0=0.0, y_lookup=None, color="#333333", label_offset=0.08):
    if y_lookup is None:
        y_lookup = assign_y(node, leaf_order)
    def walk(current, x):
        y = y_lookup[id(current)]
        if current.is_leaf:
            text = label_map.get(current.name, current.name) if label_map else current.name
            ax.text(x + label_offset, y, text, va="center", ha="left", fontsize=9)
            return
        children = [(current.left, current.left_length), (current.right, current.right_length)]
        child_ys = []
        for child, length in children:
            cy = y_lookup[id(child)]
            cx = x + length
            ax.plot([x, cx], [cy, cy], color=color, lw=1.4)
            child_ys.append(cy)
            walk(child, cx)
        ax.plot([x, x], [min(child_ys), max(child_ys)], color=color, lw=1.4)
    walk(node, x0)
    ax.set_ylim(-0.6, len(leaf_order) - 0.4)
    ax.set_yticks([])
    ax.set_xlabel("branch length (DNA-difference units)")
    clean_axes(ax)
    ax.invert_yaxis()
    ax.margins(x=0.12)
    return ax


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharex=True)
orders = [
    ["dog", "wolf", "fox", "bear", "cat", "lion"],
    ["fox", "dog", "wolf", "bear", "cat", "lion"],
    ["cat", "lion", "bear", "fox", "wolf", "dog"],
]
titles = ["Rectangular layout", "Ladderized layout", "Alternative tip order"]
for ax, order, title in zip(axes, orders, titles):
    draw_tree(ax, mammal_tree(rotated=False), order)
    ax.set_title(title, loc="left")
    ax.text(0.02, 1.02, "Which two tips are closest relatives?", transform=ax.transAxes, ha="left", va="bottom", fontsize=9, color="#4D4D4D")
fig.suptitle("The same mammal tree can look different without changing relationships.", x=0.01, ha="left", fontsize=12)
add_caption(fig, "Answer: dog and wolf are sister taxa in all three layouts; reading left-to-right across tip order is not enough.")
plt.tight_layout()
plt.show()


In all three drawings, dog and wolf remain sister taxa because they share the most recent common ancestor with each other. Tip order alone can trick your eye; the branching pattern is what carries the evidence.

Think: If a tree is rotated around an internal node, what should stay the same?


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharex=True)
draw_tree(axes[0], mammal_tree(rotated=False), ["dog", "wolf", "fox", "bear", "cat", "lion"])
axes[0].set_title("Before node rotation", loc="left")
draw_tree(axes[1], mammal_tree(rotated=True), ["fox", "wolf", "dog", "lion", "cat", "bear"])
axes[1].set_title("After rotating internal nodes", loc="left")
for ax in axes:
    ax.scatter([1.0, 3.5], [0.5, 4.5], s=42, facecolor="white", edgecolor=OKABE_ITO["green"], zorder=3)
    ax.text(0.03, 0.02, "Tips = living organisms\nInternal nodes = inferred common ancestors", transform=ax.transAxes, ha="left", va="bottom", fontsize=9, color="#4D4D4D")
axes[0].annotate("MRCA of dog and wolf", xy=(1.0, 0.5), xytext=(1.7, 1.6), arrowprops=dict(arrowstyle="-", color="#666666"), fontsize=9)
axes[0].annotate("Sister groups", xy=(3.5, 4.5), xytext=(3.7, 3.6), arrowprops=dict(arrowstyle="-", color="#666666"), fontsize=9)
fig.suptitle("Rotating a node changes the drawing, not the ancestry hypothesis.", x=0.01, ha="left", fontsize=12)
add_caption(fig, "Branch length is read from the horizontal axis here; in this notebook it means DNA difference, not time.")
plt.tight_layout()
plt.show()


*A tip is a present-day organism or sequence. An internal node is an inferred common ancestor, not a sample whose DNA we directly measured. A sister group is a pair of lineages that share an immediate common ancestor; the MRCA is the most recent common ancestor for the taxa you are tracing.*


## Section 3: Atacama dataset story

The rest of the notebook uses real 16S amplicon data from the QIIME 2 Atacama soil tutorial and its 2024.10 Atacama teaching artifacts. These are soil samples collected across a humidity and aridity gradient, with metadata recording whether vegetation was present near each sample.


## Section 4: What is an ASV?

An **ASV** is a precise DNA sequence pattern found after cleaning 16S sequencing reads. An ASV is not automatically a species.

Keep three ideas separate: **abundance** means how much of an ASV is in a sample; **taxonomic match** means what known group the sequence resembles; **evolutionary relatedness** means how sequences cluster in a tree.


## Section 5: Load cached Atacama data

The notebook starts from prepared Atacama tables so every student sees the same data when pressing Run all. The student-facing summary below shows the scale of the dataset without dumping raw metadata.


In [ ]:
summary = pd.DataFrame([
    {"What": "Samples", "Value": f"{len(metadata)}", "Why it matters": "Each row is one soil sample."},
    {"What": "ASVs available for tests", "Value": f"{len(q_asvs)}", "Why it matters": "Enough ASVs to ask which patterns track humidity or vegetation."},
    {"What": "Tree ASVs", "Value": f"{len(top12_asvs)}", "Why it matters": "Small enough for readable tip labels."},
    {"What": "Metadata used", "Value": "humidity, vegetation", "Why it matters": "These describe the soil environment."},
    {"What": "Taxonomic names", "Value": manifest["taxonomy_note"], "Why it matters": "Names are closest-reference labels, not proof of exact species."},
])
display(styled_table(summary, width_px=940))


*The dataset has enough samples to compare dry and wetter soils, but the tree will use a smaller ASV subset so the labels stay readable.*


## Section 6: Alignment of representative ASV sequences

Alignment means putting DNA letters into comparable columns. Conserved columns are mostly the same across ASVs; variable columns are where sequence differences become visible.


In [ ]:
alignment_asvs = top8_asvs
matrix = np.array([[base for base in sequences[asv]] for asv in alignment_asvs])

def identity_fraction(column):
    bases = [base for base in column if base in "ACGT"]
    if not bases:
        return 1.0
    counts = Counter(bases)
    return max(counts.values()) / len(bases)

window_width = 60
best_start = 0
best_score = -1
for start in range(0, matrix.shape[1] - window_width + 1):
    window = matrix[:, start:start + window_width]
    identity = np.array([identity_fraction(window[:, col]) for col in range(window_width)])
    variable = np.sum(identity < 0.875)
    conserved = np.sum(identity >= 0.875)
    gap_penalty = np.sum(window == "-")
    score = variable + 0.12 * conserved - 0.2 * gap_penalty
    if variable > 0 and score > best_score:
        best_start = start
        best_score = score

window = matrix[:, best_start:best_start + window_width]
identity = np.array([identity_fraction(window[:, col]) for col in range(window_width)])
alpha_by_column = np.where(identity >= 0.875, 0.30, 1.0)

fig_height = 0.5 * len(alignment_asvs) + 1.2
fig, ax = plt.subplots(figsize=(12, fig_height))
for row_idx, asv in enumerate(alignment_asvs):
    for col_idx, base in enumerate(window[row_idx]):
        color = BASE_COLORS.get(base, "#EEEEEE")
        ax.add_patch(patches.Rectangle((col_idx, row_idx), 1, 1, facecolor=color, edgecolor="none", alpha=float(alpha_by_column[col_idx])))

labels = [short_label(asv) for asv in alignment_asvs]
ax.set_xlim(0, window_width)
ax.set_ylim(0, len(alignment_asvs))
ax.set_yticks(np.arange(len(alignment_asvs)) + 0.5)
ax.set_yticklabels(labels, fontfamily="DejaVu Sans Mono", fontsize=10)
for label in ax.get_yticklabels():
    label.set_horizontalalignment("right")
ax.tick_params(axis="y", pad=18, length=0)
tick_positions = [0.5, window_width / 2, window_width - 0.5]
tick_labels = [best_start + 1, best_start + window_width // 2, best_start + window_width]
ax.set_xticks(tick_positions)
ax.set_xticklabels(tick_labels)
ax.tick_params(axis="x", top=False, bottom=True, length=3)
ax.set_xlabel("aligned marker-window column")
ax.set_title("Aligned 16S marker window - variable columns carry the phylogenetic signal.", loc="left")
ax.invert_yaxis()
for spine in ax.spines.values():
    spine.set_visible(False)
add_caption(fig, "Colored cells = DNA bases (A blue, C green, G orange, T pink). Faded columns = conserved across all sequences; bright columns = where the sequences differ.")
plt.tight_layout()
plt.show()


*ASVs that look similar across the bright columns are more closely related. The phylogenetic tree (section 8) formalizes this intuition.*


## Section 7: Distance matrix

Small distance means more similar DNA sequence. Here, each number compares two ASVs in the same aligned 16S marker region.


In [ ]:
distance_asvs = top12_asvs
distance_matrix = pd.DataFrame(index=distance_asvs, columns=distance_asvs, dtype=float)
for a in distance_asvs:
    for b in distance_asvs:
        distance_matrix.loc[a, b] = sequence_distance(sequences[a], sequences[b])

fig, ax = plt.subplots(figsize=(7.2, 6.1))
shown = distance_matrix.loc[distance_asvs, distance_asvs]
im = ax.imshow(shown.values, cmap="viridis", vmin=0, vmax=np.nanmax(shown.values))
ax.set_xticks(range(len(distance_asvs)))
ax.set_yticks(range(len(distance_asvs)))
ax.set_xticklabels([short_label(x).replace("Atacama ", "") for x in distance_asvs], rotation=45, ha="right")
ax.set_yticklabels([short_label(x).replace("Atacama ", "") for x in distance_asvs])
if len(distance_asvs) <= 12:
    for i in range(len(distance_asvs)):
        for j in range(len(distance_asvs)):
            ax.text(j, i, f"{shown.values[i, j]:.2f}", ha="center", va="center", fontsize=7, color="white" if shown.values[i, j] > 0.18 else "#222222")
ax.set_title("Pairwise DNA distance among top Atacama ASVs.", loc="left")
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("sequence-distance units")
clean_axes(ax)
add_caption(fig, "Small distances mark ASV sequences with more similar DNA letters in comparable columns.")
plt.tight_layout()
plt.show()


*The darkest cells mark the smallest distances; those pairs are the first candidates for close sequence relatedness.*


## Section 8: UPGMA tree (the payoff)

Tips are ASVs. Branch length shows sequence difference. Branches that meet recently, with a short path between them, suggest closer sequence relatedness, which is our best evidence of evolutionary relatedness here but not proof of exact species identity.


In [ ]:
@dataclass
class UPGMANode:
    name: str | None = None
    left: object | None = None
    right: object | None = None
    height: float = 0.0
    left_length: float = 0.0
    right_length: float = 0.0
    members: tuple[str, ...] = ()

    @property
    def is_leaf(self):
        return self.name is not None

def build_upgma(distance_df):
    clusters = {name: UPGMANode(name=name, height=0.0, members=(name,)) for name in distance_df.index}
    next_id = 1
    while len(clusters) > 1:
        keys = list(clusters)
        best = None
        best_distance = math.inf
        for i, a in enumerate(keys):
            for b in keys[i + 1:]:
                vals = [distance_df.loc[x, y] for x in clusters[a].members for y in clusters[b].members]
                d = float(np.mean(vals))
                if d < best_distance:
                    best = (a, b)
                    best_distance = d
        a, b = best
        left = clusters.pop(a)
        right = clusters.pop(b)
        height = best_distance / 2.0
        node = UPGMANode(
            left=left,
            right=right,
            height=height,
            left_length=max(0.0, height - left.height),
            right_length=max(0.0, height - right.height),
            members=tuple(sorted(left.members + right.members)),
        )
        clusters[f"node_{next_id}"] = node
        next_id += 1
    return next(iter(clusters.values()))

def leaves_in_order(node):
    if node.is_leaf:
        return [node.name]
    return leaves_in_order(node.left) + leaves_in_order(node.right)

def newick(node):
    if node.is_leaf:
        return node.name
    return f"({newick(node.left)}:{node.left_length:.6f},{newick(node.right)}:{node.right_length:.6f})"

upgma_tree = build_upgma(distance_matrix)
tree_newick = newick(upgma_tree) + ";"
leaf_order = leaves_in_order(upgma_tree)
y_lookup = {name: i for i, name in enumerate(leaf_order)}

def node_y(node):
    if node.is_leaf:
        return y_lookup[node.name]
    return (node_y(node.left) + node_y(node.right)) / 2

tax_lookup = feature_key.set_index("asv")["closest_taxonomic_match"].to_dict()
fig_height = max(4.8, 0.48 * len(leaf_order) + 1.5)
fig, ax = plt.subplots(figsize=(9.5, fig_height))

def draw_upgma(node, x):
    y = node_y(node)
    if node.is_leaf:
        match = tax_lookup.get(node.name, "Unassigned at genus level")
        suffix = "unassigned" if match == "Unassigned at genus level" else match
        ax.text(x + 0.01, y, f"{short_label(node.name)} ({suffix})", va="center", ha="left", fontsize=9)
        return
    children = [(node.left, node.left_length), (node.right, node.right_length)]
    child_ys = []
    for child, length in children:
        cy = node_y(child)
        cx = x + length
        ax.plot([x, cx], [cy, cy], color="#333333", lw=1.3)
        child_ys.append(cy)
        draw_upgma(child, cx)
    ax.plot([x, x], [min(child_ys), max(child_ys)], color="#333333", lw=1.3)

draw_upgma(upgma_tree, 0.0)
ax.set_ylim(-0.6, len(leaf_order) - 0.4)
ax.invert_yaxis()
ax.set_yticks([])
ax.set_xlabel("branch length (sequence-distance units)")
ax.set_title("UPGMA tree for the top Atacama ASVs.", loc="left")
clean_axes(ax)
ax.margins(x=0.25)
ax.set_xlim(left=0)
add_caption(fig, "Tips are ASVs. Short paths between tips suggest closer sequence relatedness, not exact species identity.")
plt.tight_layout()
plt.show()


*This UPGMA tree turns the distance matrix into a readable hypothesis: nearby tips have more similar 16S sequences than tips far apart on the tree.*


## Section 9: Relative abundance - what is actually in these samples?

The tree asks how sequences are related. Abundance asks a different question: which ASVs make up each soil sample, and does that composition change from drier to wetter soils?


In [ ]:
abundance_table = (
    feature_key.query("in_abundance_top20 == True")
    .loc[:, ["asv", "closest_taxonomic_match", "mean_relative_abundance_percent", "max_relative_abundance_percent", "prevalence_samples"]]
    .rename(columns={
        "asv": "ASV",
        "closest_taxonomic_match": "Closest taxonomic match",
        "mean_relative_abundance_percent": "Mean relative abundance (%)",
        "max_relative_abundance_percent": "Maximum relative abundance (%)",
        "prevalence_samples": "Samples detected (of 61)",
    })
)
abundance_table["Mean relative abundance (%)"] = abundance_table["Mean relative abundance (%)"].round(2)
abundance_table["Maximum relative abundance (%)"] = abundance_table["Maximum relative abundance (%)"].round(2)
display(styled_table(abundance_table, width_px=1000))


In [ ]:
plot_df = relative_top20.merge(metadata[["sample_id", "average_soil_relative_humidity", "vegetation"]], on="sample_id")
plot_df = plot_df.sort_values("average_soil_relative_humidity").reset_index(drop=True)
stack_cols = [col for col in top20_asvs if col in plot_df.columns] + ["Other"]

fig, ax = plt.subplots(figsize=(12, 4.6))
bottom = np.zeros(len(plot_df))
x = np.arange(len(plot_df))
for col in stack_cols:
    values = plot_df[col].to_numpy(dtype=float)
    ax.bar(x, values, bottom=bottom, width=0.88, color=asv_color.get(col, "#BDBDBD"), edgecolor="white", linewidth=0.15, label=short_label(col))
    bottom += values

ax.set_ylim(0, 100)
ax.set_xlim(-0.5, len(plot_df) - 0.5)
ax.set_ylabel("relative abundance (%)")
ax.set_xlabel("samples ordered from drier to wetter soil")
ax.set_xticks([])
ax.set_title("Top ASVs across Atacama soil samples ordered by humidity.", loc="left")
ax.text(0, -9, "drier", ha="left", va="top", fontsize=9, color="#4D4D4D")
ax.text(len(plot_df) - 1, -9, "wetter", ha="right", va="top", fontsize=9, color="#4D4D4D")
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:10], labels[:10], bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8, title="first 10 ASVs")
clean_axes(ax)
add_caption(fig, "Each vertical bar is one soil sample; color shows which ASVs make up the sample after the top 20 are separated from Other.")
plt.tight_layout()
plt.show()


*A very abundant ASV is not automatically the most evolutionarily unusual one; abundance and relatedness answer different questions.*


## Section 10: Alpha diversity

Alpha diversity asks how diverse one sample is: how many ASV types are present, and how evenly distributed they are. We use observed ASVs for richness and Shannon diversity for richness plus evenness.


In [ ]:
alpha_plot = alpha_diversity.merge(metadata[["sample_id", "average_soil_relative_humidity", "vegetation"]], on="sample_id")
alpha_summary = (
    alpha_plot.groupby("vegetation", as_index=False)
    .agg(
        Samples=("sample_id", "count"),
        **{
            "Mean observed ASVs": ("observed_asvs", "mean"),
            "Mean Shannon diversity": ("shannon_diversity", "mean"),
            "Mean humidity (%)": ("average_soil_relative_humidity", "mean"),
        },
    )
)
alpha_summary["Mean observed ASVs"] = alpha_summary["Mean observed ASVs"].round(1)
alpha_summary["Mean Shannon diversity"] = alpha_summary["Mean Shannon diversity"].round(2)
alpha_summary["Mean humidity (%)"] = alpha_summary["Mean humidity (%)"].round(1)
alpha_summary = alpha_summary.rename(columns={"vegetation": "Vegetation"})
display(styled_table(alpha_summary, width_px=760))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8), sharex=True)
metrics = [("observed_asvs", "observed ASVs"), ("shannon_diversity", "Shannon diversity")]
for ax, (metric, label) in zip(axes, metrics):
    x = alpha_plot["average_soil_relative_humidity"].to_numpy(dtype=float)
    y = alpha_plot[metric].to_numpy(dtype=float)
    ax.scatter(x, y, s=28, color=OKABE_ITO["blue"], alpha=0.78, edgecolor="white", linewidth=0.3)
    valid = np.isfinite(x) & np.isfinite(y)
    if valid.sum() >= 3:
        fit = np.polyfit(x[valid], y[valid], 1)
        xs = np.linspace(np.nanmin(x), np.nanmax(x), 100)
        ax.plot(xs, fit[0] * xs + fit[1], color="#666666", lw=1.2, alpha=0.7)
    ax.set_xlabel("average soil relative humidity (%)")
    ax.set_ylabel(label)
    ax.set_title(label, loc="left")
    clean_axes(ax)
fig.suptitle("Alpha diversity across the humidity gradient.", x=0.01, ha="left", fontsize=12)
add_caption(fig, "Observed ASVs count types; Shannon diversity increases when a sample has many types with more even abundances.")
plt.tight_layout()
plt.show()


*Use the trend lines as a visual guide, then describe the direction carefully: humid samples may be richer or more even, but the plot does not identify exact species.*


## Section 11: BH-corrected association tests

We tested 50 ASVs to see if abundance changes with humidity. With a p < 0.05 cutoff, we would expect about 50 x 0.05 = 2.5 false positives by random chance alone, even if nothing is truly associated. Benjamini-Hochberg (BH) correction adjusts for this. A q-value of 0.05 means we expect about 5% of discoveries below that threshold to be false alarms.


In [ ]:
count_cols = [col for col in counts_top50.columns if col.startswith("Atacama_ASV_")]
q_counts = counts_top50[["sample_id", *count_cols]].merge(metadata[["sample_id", "average_soil_relative_humidity", "vegetation"]], on="sample_id")
raw_counts = q_counts[count_cols].astype(float)
# CLR uses a 0.5 pseudo-count so zero counts can be logged without creating infinite values.
logged = np.log(raw_counts + 0.5)
clr = logged.sub(logged.mean(axis=1), axis=0)
sample_totals = counts_top50.set_index("sample_id")["total_reads"].reindex(q_counts["sample_id"]).to_numpy(dtype=float)
rel = raw_counts.div(sample_totals, axis=0) * 100.0

humidity = q_counts["average_soil_relative_humidity"].astype(float)
vegetation = q_counts["vegetation"].astype(str).str.lower()
rows = []
for asv in count_cols:
    rho, p_humidity = stats.spearmanr(clr[asv], humidity, nan_policy="omit")
    yes_values = clr.loc[vegetation == "yes", asv]
    no_values = clr.loc[vegetation == "no", asv]
    if len(yes_values) > 0 and len(no_values) > 0:
        u_stat, p_veg = stats.mannwhitneyu(yes_values, no_values, alternative="two-sided")
    else:
        p_veg = np.nan
    mean_yes = rel.loc[vegetation == "yes", asv].mean()
    mean_no = rel.loc[vegetation == "no", asv].mean()
    log2_fc = float(np.log2((mean_yes + 0.001) / (mean_no + 0.001)))
    mean_abundance = rel[asv].mean()
    rows.append({"ASV": asv, "Variable": "Humidity", "Effect": float(rho), "Raw p-value": float(p_humidity), "Mean abundance (%)": float(mean_abundance)})
    rows.append({"ASV": asv, "Variable": "Vegetation", "Effect": log2_fc, "Raw p-value": float(p_veg), "Mean abundance (%)": float(mean_abundance)})

association_results = pd.DataFrame(rows)
q_values = []
for variable, group in association_results.groupby("Variable", sort=False):
    _, adjusted, _, _ = multipletests(group["Raw p-value"].fillna(1.0).to_numpy(dtype=float), method="fdr_bh")
    q_values.extend(pd.Series(adjusted, index=group.index).items())
q_lookup = dict(q_values)
association_results["BH q-value"] = association_results.index.map(q_lookup).astype(float)
association_results = association_results.merge(
    feature_key[["asv", "closest_taxonomic_match", "phylum", "genus", "prevalence_samples"]].rename(columns={"asv": "ASV"}),
    on="ASV",
    how="left",
)
association_results["Significant?"] = np.where(association_results["BH q-value"] < 0.05, "yes", "-")

humidity_results = association_results.query("Variable == 'Humidity'").copy()
fig, axes = plt.subplots(1, 2, figsize=(4, 3), sharey=True)
rng = np.random.default_rng(7)
for ax, col, title in zip(axes, ["Raw p-value", "BH q-value"], ["Raw p-values", "BH q-values"]):
    y = humidity_results[col].to_numpy(dtype=float)
    x = rng.normal(0, 0.035, size=len(y))
    ax.scatter(x, y, s=18, color=OKABE_ITO["blue"], alpha=0.62, edgecolor="none")
    ax.axhline(0.05, color="#666666", ls="--", lw=1)
    ax.set_xlim(-0.18, 0.18)
    ax.set_xticks([])
    ax.set_title(title, loc="left", fontsize=10)
    clean_axes(ax)
axes[0].set_ylabel("value")
fig.suptitle("Raw p-values compared with BH q-values.", x=0.02, ha="left", fontsize=11)
add_caption(fig, "Multiple-testing correction is conservative on purpose - it is the cost of asking many questions at once.")
plt.tight_layout()
plt.show()


In [ ]:
table_rows = []
for variable in ["Humidity", "Vegetation"]:
    subset = association_results.query("Variable == @variable").sort_values("BH q-value").head(10).copy()
    for _, row in subset.iterrows():
        if variable == "Humidity":
            effect_label = f"{row['Effect']:.2f}"
            effect_col = "Spearman rho"
        else:
            arrow = "higher with vegetation" if row["Effect"] > 0 else "higher without vegetation"
            effect_label = f"{row['Effect']:.2f} ({arrow})"
            effect_col = "Effect size"
        table_rows.append({
            "Metadata variable": variable,
            "ASV": row["ASV"],
            "Closest taxonomic match": row["closest_taxonomic_match"],
            "Mean abundance (%)": round(row["Mean abundance (%)"], 2),
            "Samples detected (of 61)": int(row["prevalence_samples"]),
            "Spearman rho": effect_label if variable == "Humidity" else "",
            "Effect size": effect_label if variable == "Vegetation" else "",
            "Raw p-value": fmt_p(row["Raw p-value"]),
            "BH q-value": fmt_p(row["BH q-value"]),
            "Significant?": row["Significant?"],
        })

q_table = pd.DataFrame(table_rows)

def significant_style(row):
    if row["Significant?"] == "yes":
        return ["border-left: 3px solid #009E73"] + [""] * (len(row) - 1)
    return [""] * len(row)

def sig_color(value):
    if value == "yes":
        return "color: #009E73; font-weight: 700"
    if value == "-":
        return "color: #999999"
    return ""

def q_background(value):
    try:
        numeric = float(str(value).replace("e", "E"))
    except ValueError:
        return ""
    strength = 1.0 - min(max(numeric / 0.05, 0.0), 1.0)
    return f"background-color: rgba(0, 158, 115, {0.06 + 0.18 * strength:.3f})"

display(
    q_table.style.hide(axis="index")
    .apply(significant_style, axis=1)
    .map(sig_color, subset=["Significant?"])
    .map(q_background, subset=["BH q-value"])
    .set_table_styles([
        {"selector": "table", "props": [("border-collapse", "collapse"), ("width", "1080px"), ("font-size", "12px")]},
        {"selector": "th", "props": [("text-align", "left"), ("border-bottom", "1px solid #999"), ("padding", "6px 7px")]},
        {"selector": "td", "props": [("padding", "6px 7px"), ("border-bottom", "1px solid #e6e6e6")]},
        {"selector": "tbody tr:nth-child(odd)", "props": [("background-color", "#fafafa")]},
    ])
)


In [ ]:
# The full top-50 test table includes low-prevalence ASVs, but the teaching plot
# only labels discoveries that meet the original >=10% prevalence threshold.
min_prevalence_for_plot = int(manifest["present_threshold_samples"])
significant = association_results.query("`BH q-value` < 0.05 and prevalence_samples >= @min_prevalence_for_plot").copy()
panels = ["Humidity", "Vegetation"]
fig, axes = plt.subplots(1, 2, figsize=(11, max(3.4, 0.38 * max(1, significant.groupby("Variable").size().max() if not significant.empty else 1) + 1.4)))
for ax, variable in zip(axes, panels):
    subset = significant.query("Variable == @variable").copy()
    if subset.empty:
        ax.text(0.5, 0.5, "No ASVs below q < 0.05", ha="center", va="center", transform=ax.transAxes, color="#666666")
        ax.set_axis_off()
        continue
    subset = subset.sort_values("Effect")
    y = np.arange(len(subset))
    colors = [OKABE_ITO["blue"] if row["phylum"] in ("", "Unassigned") else OKABE_ITO["green"] for _, row in subset.iterrows()]
    sizes = 45 + 18 * np.sqrt(subset["Mean abundance (%)"].clip(lower=0.01))
    ax.axvline(0, color="#777777", lw=1)
    ax.hlines(y, 0, subset["Effect"], color="#777777", lw=1.1)
    ax.scatter(subset["Effect"], y, s=sizes, color=colors, alpha=0.88, edgecolor="white", linewidth=0.4, zorder=3)
    labels = []
    for _, row in subset.iterrows():
        match = row["genus"] if isinstance(row["genus"], str) and row["genus"] else row.get("closest_taxonomic_match", "unassigned")
        if not isinstance(match, str) or not match:
            match = "unassigned"
        labels.append(f"{short_label(row['ASV'])} ({match})")
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=8)
    x_right = max(subset["Effect"].max(), 0) + 0.08
    for yi, (_, row) in enumerate(subset.iterrows()):
        ax.text(x_right, yi, f"q={fmt_p(row['BH q-value'])}", ha="left", va="center", fontsize=8, color="#999999")
    ax.set_xlabel("Spearman rho" if variable == "Humidity" else "log2 fold-change")
    ax.set_title(variable, loc="left")
    clean_axes(ax)
    ax.margins(x=0.22)
fig.suptitle("ASVs with BH q-values below 0.05 and enough prevalence to interpret.", x=0.01, y=0.98, ha="left", fontsize=12)
add_caption(fig, "Each bar is one ASV. Humidity bars to the right increase with wetter soil; vegetation bars to the right are higher with vegetation. Dot size = overall abundance. Only ASVs with BH q < 0.05 and >=10% prevalence shown.")
plt.tight_layout(rect=[0, 0.06, 1, 0.91])
plt.show()


*BH q-values summarize abundance-versus-metadata tests. They are not tree branch support values, and they do not say an ASV is a proved species.*


## Section 12: Final student report

Fill in the report using the section numbers named in each question.

1. Which 3 ASVs are most abundant in your samples? Refer to section 9 table.  
   [your answer]

2. Which ASVs are significantly associated with humidity? Refer to section 11 lollipop.  
   [your answer]

3. Which ASVs are significantly associated with vegetation? Refer to section 11 lollipop.  
   [your answer]

4. What does alpha diversity suggest about humid versus arid samples? Refer to section 10.  
   [your answer]

5. In the UPGMA tree, which ASVs cluster closest together? Refer to section 8.  
   [your answer]

6. Are closely related ASVs from section 8 also similar in abundance from section 9 or in humidity association from section 11?  
   [your answer]

7. Synthesis: An ASV can be (a) very abundant, (b) statistically associated with humidity, and (c) closely related to another ASV in the tree - and these are three different things. Explain in 2-3 sentences why these are three different ideas and why all three matter.  
   [your answer]
